# 04 — Call Summarization

Generate structured summaries from 12 customer service calls using ASR transcripts and speaker diarization results.

**Pipeline:**
1. Load processed transcripts.
2. Prepare speaker-attributed conversations.
3. Generate structured call summaries.
4. Evaluate summary quality.
5. Export results.

## 1. Input Data Inspection

Check the availability of processed ASR and diarization files from previous stages.

In [1]:
from pathlib import Path

print("=" * 60)
print("STAGE 04 — INPUT DATA INSPECTION")
print("=" * 60)

print(f"\nCurrent directory: {Path.cwd()}")

directories = [
    Path("data/processed/asr"),
    Path("data/processed/diarization"),
    Path("data/processed"),
]

for directory in directories:

    print(f"\nDirectory: {directory}")

    if directory.exists():

        files = [
            p for p in directory.rglob("*")
            if p.is_file() and p.suffix.lower() in [".json", ".csv"]
        ]

        print(f"Files found: {len(files)}")

        for file in files[:15]:
            print(f"  {file}")

    else:
        print("Directory not found.")

print("\nAvailable ZIP files:")

zip_files = list(Path(".").glob("*.zip"))

for file in zip_files[:15]:
    print(f"  {file}")

print("\nInspection completed.")

STAGE 04 — INPUT DATA INSPECTION

Current directory: /content

Directory: data/processed/asr
Directory not found.

Directory: data/processed/diarization
Directory not found.

Directory: data/processed
Directory not found.

Available ZIP files:

Inspection completed.


## 2. Restore Processed Data

Load previously generated ASR and diarization results to avoid redundant inference.

In [2]:
from google.colab import files
from pathlib import Path
import zipfile

print("=" * 60)
print("STAGE 04 — RESTORE PREVIOUS RESULTS")
print("=" * 60)

print("\nUpload ASR and Diarization ZIP files:")

uploaded = files.upload()

for filename in uploaded:

    if filename.lower().endswith(".zip"):

        print(f"\nExtracting: {filename}")

        with zipfile.ZipFile(filename, "r") as zip_ref:
            zip_ref.extractall("/content")

        print(f"Successfully extracted: {filename}")

    else:
        print(f"Skipped non-ZIP file: {filename}")

print("\n" + "=" * 60)
print("AVAILABLE JSON AND CSV FILES")
print("=" * 60)

result_files = [
    p for p in Path("/content").rglob("*")
    if p.is_file()
    and p.suffix.lower() in [".json", ".csv"]
]

print(f"\nTotal files found: {len(result_files)}")

for file in result_files[:60]:
    print(file.relative_to("/content"))

print("\nRestoration completed.")

STAGE 04 — RESTORE PREVIOUS RESULTS

Upload ASR and Diarization ZIP files:


Saving Maharah_Diarization_Results.zip to Maharah_Diarization_Results.zip
Saving Maharah_Processed_Data.zip to Maharah_Processed_Data.zip

Extracting: Maharah_Diarization_Results.zip
Successfully extracted: Maharah_Diarization_Results.zip

Extracting: Maharah_Processed_Data.zip
Successfully extracted: Maharah_Processed_Data.zip

AVAILABLE JSON AND CSV FILES

Total files found: 55
.config/.last_update_check.json
sample_data/anscombe.json
sample_data/mnist_test.csv
sample_data/mnist_train_small.csv
sample_data/california_housing_train.csv
sample_data/california_housing_test.csv
data/processed/selected_calls.csv
data/processed/diarization/community_1/CALL_009_turns.csv
data/processed/diarization/community_1/CALL_005_turns.csv
data/processed/diarization/community_1/CALL_008.json
data/processed/diarization/community_1/CALL_007.json
data/processed/diarization/community_1/CALL_006_turns.csv
data/processed/diarization/community_1/CALL_004_turns.csv
data/processed/diarization/community_1/CALL_0

## 3. Data Structure Inspection

Inspect transcript segments, speaker labels, timestamps, and call metadata to verify input compatibility.

In [3]:
import json
import pandas as pd
from pathlib import Path

ASR_DIR = Path("data/processed/asr/small_en_cpu")
DIAR_DIR = Path("data/processed/diarization/community_1")

CALL_ID = "CALL_001"

print("=" * 65)
print("STAGE 04 — DATA STRUCTURE INSPECTION")
print("=" * 65)

# --------------------------------------------------
# 1. Inspect ASR results
# --------------------------------------------------

asr_path = ASR_DIR / f"{CALL_ID}.json"

with open(asr_path, "r", encoding="utf-8") as f:
    asr_data = json.load(f)

print("\n[1] WHISPER ASR STRUCTURE")
print("-" * 65)

print("Data type:", type(asr_data).__name__)

if isinstance(asr_data, dict):
    print("Available keys:", list(asr_data.keys()))

    for key, value in asr_data.items():

        if isinstance(value, list):
            print(f"\n{key}: List with {len(value)} items")

            if value:
                print("First item:")
                print(json.dumps(
                    value[0],
                    indent=2,
                    ensure_ascii=False
                )[:1500])

        elif isinstance(value, dict):
            print(f"\n{key}: Dictionary")
            print("Keys:", list(value.keys()))

        else:
            print(f"{key}: {str(value)[:250]}")

# --------------------------------------------------
# 2. Inspect Diarization results
# --------------------------------------------------

diar_path = DIAR_DIR / f"{CALL_ID}.json"

with open(diar_path, "r", encoding="utf-8") as f:
    diar_data = json.load(f)

print("\n\n[2] PYANNOTE DIARIZATION STRUCTURE")
print("-" * 65)

print("Data type:", type(diar_data).__name__)

if isinstance(diar_data, dict):
    print("Available keys:", list(diar_data.keys()))

    for key, value in diar_data.items():

        if isinstance(value, list):
            print(f"\n{key}: List with {len(value)} items")

            if value:
                print("First item:")
                print(json.dumps(
                    value[0],
                    indent=2,
                    ensure_ascii=False
                )[:1500])

        elif isinstance(value, dict):
            print(f"\n{key}: Dictionary")
            print("Keys:", list(value.keys()))

        else:
            print(f"{key}: {str(value)[:250]}")

# --------------------------------------------------
# 3. Inspect Speaker Turns
# --------------------------------------------------

turns_path = DIAR_DIR / f"{CALL_ID}_turns.csv"

turns_df = pd.read_csv(turns_path)

print("\n\n[3] SPEAKER TURNS STRUCTURE")
print("-" * 65)

print("Columns:", list(turns_df.columns))
print("Total turns:", len(turns_df))

print("\nFirst 10 turns:")
print(turns_df.head(10).to_string(index=False))

# --------------------------------------------------
# 4. Inspect selected calls
# --------------------------------------------------

selected_df = pd.read_csv(
    "data/processed/selected_calls.csv"
)

print("\n\n[4] SELECTED CALLS")
print("-" * 65)

print("Columns:", list(selected_df.columns))
print("Total calls:", len(selected_df))

print("\nFirst 3 rows:")
print(selected_df.head(3).to_string(index=False))

print("\n" + "=" * 65)
print("INSPECTION COMPLETED")
print("=" * 65)

STAGE 04 — DATA STRUCTURE INSPECTION

[1] WHISPER ASR STRUCTURE
-----------------------------------------------------------------
Data type: dict
Available keys: ['call_id', 'accent', 'domain', 'audio_duration_sec', 'source_dataset', 'dataset_revision', 'model_repo', 'model_revision', 'asr_settings', 'run_utc', 'inference_seconds', 'predicted_language', 'predicted_transcript', 'predicted_segments', 'speaker_label_method']
call_id: CALL_001
accent: en-US_General
domain: food
audio_duration_sec: 386.637
source_dataset: apptek-com/apptek_callcenter_dialogues
dataset_revision: b98967d9946f7f59f58d08624a2a00fe98fe0219
model_repo: Systran/faster-whisper-small.en
model_revision: d1d751a5f8271d482d14ca55d9e2deeebbae577f

asr_settings: Dictionary
Keys: ['language', 'beam_size', 'vad_filter', 'vad_parameters', 'word_timestamps', 'condition_on_previous_text']
run_utc: 2026-11-01T14:09:27.837540+00:00
inference_seconds: 97.07
predicted_language: en
predicted_transcript: ring ring hi there this is 

## 4. Summarization Input Preparation

Convert ASR-aligned speaker turns into structured transcripts while preserving conversation order and speaker roles.

**Outputs:**
- Individual call JSON files
- Combined transcript dataset
- Input statistics


In [5]:
import json
import pandas as pd
from pathlib import Path

print("=" * 65)
print("STAGE 04 — PREPARE SUMMARIZATION INPUT")
print("=" * 65)

# 1. Define paths
ASR_DIR = Path("data/processed/asr/small_en_cpu")
DIAR_DIR = Path("data/processed/diarization/community_1")
OUTPUT_DIR = Path("data/processed/summarization/input")

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

selected_calls = pd.read_csv(
    "data/processed/selected_calls.csv"
)

all_calls = []
statistics = []

# 2. Process each call
for call_id in selected_calls["call_id"]:

    asr_path = ASR_DIR / f"{call_id}.json"
    diar_path = DIAR_DIR / f"{call_id}.json"
    turns_path = DIAR_DIR / f"{call_id}_turns.csv"

    with open(asr_path, "r", encoding="utf-8") as f:
        asr_data = json.load(f)

    with open(diar_path, "r", encoding="utf-8") as f:
        diar_data = json.load(f)

    turns_df = pd.read_csv(turns_path)

    required_columns = {"start", "end", "speaker", "role", "text"}

    if not required_columns.issubset(turns_df.columns):
        raise ValueError(f"{call_id}: Missing required columns")

    turns_df = turns_df.sort_values(
        ["start", "end"]
    ).reset_index(drop=True)

    formatted_turns = []
    agent_turns = 0
    customer_turns = 0
    unknown_turns = 0

    for _, row in turns_df.iterrows():

        role = str(row["role"]).strip().lower()
        text = str(row["text"]).strip()

        if not text or text.lower() == "nan":
            continue

        if role == "agent":
            speaker = "Agent"
            agent_turns += 1

        elif role == "customer":
            speaker = "Customer"
            customer_turns += 1

        else:
            speaker = str(row["speaker"])
            unknown_turns += 1

        formatted_turns.append({
            "start": float(row["start"]),
            "end": float(row["end"]),
            "speaker": speaker,
            "text": text
        })

    # 3. Build formatted transcript
    transcript_lines = []

    for turn in formatted_turns:
        transcript_lines.append(
            f"[{turn['start']:.2f}s - {turn['end']:.2f}s] "
            f"{turn['speaker']}: {turn['text']}"
        )

    formatted_transcript = "\n".join(transcript_lines)

    # 4. Create structured record
    call_record = {
        "call_id": call_id,
        "domain": asr_data["domain"],
        "accent": asr_data["accent"],
        "duration_sec": asr_data["audio_duration_sec"],
        "asr_model": asr_data["model_repo"],
        "diarization_model": diar_data["model"],
        "role_label_source": diar_data.get(
            "role_label_source", "unknown"
        ),
        "speaker_turns": formatted_turns,
        "formatted_transcript": formatted_transcript
    }

    all_calls.append(call_record)

    with open(
        OUTPUT_DIR / f"{call_id}.json",
        "w",
        encoding="utf-8"
    ) as f:
        json.dump(call_record, f, indent=2, ensure_ascii=False)

    # 5. Collect statistics
    statistics.append({
        "call_id": call_id,
        "domain": asr_data["domain"],
        "duration_sec": asr_data["audio_duration_sec"],
        "total_turns": len(formatted_turns),
        "agent_turns": agent_turns,
        "customer_turns": customer_turns,
        "unknown_turns": unknown_turns,
        "word_count": len(formatted_transcript.split())
    })

# 6. Save combined results
with open(
    OUTPUT_DIR / "all_calls.json",
    "w",
    encoding="utf-8"
) as f:
    json.dump(all_calls, f, indent=2, ensure_ascii=False)

stats_df = pd.DataFrame(statistics)

stats_df.to_csv(
    OUTPUT_DIR / "input_statistics.csv",
    index=False
)

# 7. Display results
print("\nSUMMARIZATION INPUT STATISTICS")
print("-" * 65)

display(stats_df)

print("\n" + "=" * 65)
print("DATASET SUMMARY")
print("=" * 65)

print(f"Total calls: {len(all_calls)}")
print(f"Total speaker turns: {stats_df['total_turns'].sum()}")
print(f"Total words: {stats_df['word_count'].sum():,}")
print(f"Unknown speaker turns: {stats_df['unknown_turns'].sum()}")

print(f"\nOutput directory: {OUTPUT_DIR}")

print("\n" + "=" * 65)
print("SAMPLE TRANSCRIPT — CALL_001")
print("=" * 65)

print(all_calls[0]["formatted_transcript"][:2500])

print("\n" + "=" * 65)
print("INPUT PREPARATION COMPLETED")
print("=" * 65)

STAGE 04 — PREPARE SUMMARIZATION INPUT

SUMMARIZATION INPUT STATISTICS
-----------------------------------------------------------------


,call_id,domain,duration_sec,total_turns,agent_turns,customer_turns,unknown_turns,word_count
0,CALL_001,food,386.637,31,14,17,0,877
1,CALL_002,hospitality,630.749,48,22,26,0,1610
2,CALL_003,entertainment,701.906,97,51,46,0,1523
3,CALL_004,hospitality,421.525,51,28,23,0,1292
4,CALL_005,retail,611.108,78,51,27,0,1920
5,CALL_006,entertainment,603.482,66,32,34,0,1398
6,CALL_007,telecom,631.712,105,58,47,0,1554
7,CALL_008,insurance,625.664,77,43,34,0,1553
8,CALL_009,health,422.208,66,36,30,0,1028
9,CALL_010,banking,420.000,52,34,18,0,1080



DATASET SUMMARY
Total calls: 12
Total speaker turns: 776
Total words: 18,364
Unknown speaker turns: 0

Output directory: data/processed/summarization/input

SAMPLE TRANSCRIPT — CALL_001
[11.44s - 19.74s] Agent: ring ring hi there this is Leo from feast and co your personal lunch superhero am I speaking with Ashley
[19.74s - 26.28s] Customer: hi Leo yeah this is Ashley I'm looking into corporate catering for a team lunch
[26.28s - 37.28s] Agent: oh I love a good team lunch nothing brings co -workers together like gourmet bites and good vibes so tell me what kind of flavor adventure are we talking about here
[39.29s - 51.21s] Customer: Wow you sound very enthusiastic about this I love it well we have a few dietary restrictions some people are vegan and a couple are gluten -free
[51.21s - 65.57s] Agent: challenge accepted vegan check gluten free double check I have a mental cookbook bursting with options we can make sure everyone leaves the table happy and full
[65.57s - 72.93s] Customer

## 5. Model Selection and Hardware Check

Inspect available computational resources to select a suitable instruction-tuned LLM for structured call summarization.

**Selection criteria:** Context length, summary quality, memory requirements, and inference efficiency.

In [6]:
import torch
import psutil
import pandas as pd
from pathlib import Path

print("=" * 65)
print("STAGE 04 — MODEL SELECTION & HARDWARE CHECK")
print("=" * 65)

# 1. Environment
print("\n[1] ENVIRONMENT")
print("-" * 65)

print(f"PyTorch Version: {torch.__version__}")
print(f"CUDA Available: {torch.cuda.is_available()}")

# 2. GPU Information
print("\n[2] GPU INFORMATION")
print("-" * 65)

if torch.cuda.is_available():

    gpu_name = torch.cuda.get_device_name(0)

    gpu_memory = (
        torch.cuda.get_device_properties(0).total_memory
        / 1024**3
    )

    print(f"GPU: {gpu_name}")
    print(f"GPU Memory: {gpu_memory:.2f} GB")

else:
    print("No GPU detected — CPU runtime active.")

# 3. System RAM
print("\n[3] SYSTEM MEMORY")
print("-" * 65)

ram = psutil.virtual_memory()

print(f"Total RAM: {ram.total / 1024**3:.2f} GB")
print(f"Available RAM: {ram.available / 1024**3:.2f} GB")

# 4. Input Statistics
print("\n[4] SUMMARIZATION INPUT")
print("-" * 65)

input_dir = Path(
    "data/processed/summarization/input"
)

stats = pd.read_csv(
    input_dir / "input_statistics.csv"
)

print(f"Total Calls: {len(stats)}")
print(f"Total Words: {stats['word_count'].sum():,}")
print(f"Average Words: {stats['word_count'].mean():.0f}")
print(f"Maximum Words: {stats['word_count'].max():,}")

longest_call = stats.loc[
    stats["word_count"].idxmax()
]

print(f"Longest Call: {longest_call['call_id']}")

print("\n" + "=" * 65)
print("HARDWARE CHECK COMPLETED")
print("=" * 65)

STAGE 04 — MODEL SELECTION & HARDWARE CHECK

[1] ENVIRONMENT
-----------------------------------------------------------------
PyTorch Version: 2.11.0+cu130
CUDA Available: True

[2] GPU INFORMATION
-----------------------------------------------------------------
GPU: Tesla T4
GPU Memory: 14.56 GB

[3] SYSTEM MEMORY
-----------------------------------------------------------------
Total RAM: 12.67 GB
Available RAM: 11.06 GB

[4] SUMMARIZATION INPUT
-----------------------------------------------------------------
Total Calls: 12
Total Words: 18,364
Average Words: 1530
Maximum Words: 2,514
Longest Call: CALL_012

HARDWARE CHECK COMPLETED


## 6. Model Setup

Load Qwen2.5-3B-Instruct with 4-bit quantization for memory-efficient inference.

In [7]:
import torch

if not torch.cuda.is_available():
    raise RuntimeError(
        "GPU not detected. Enable T4 GPU in Colab before proceeding."
    )

print("GPU:", torch.cuda.get_device_name(0))
print("CUDA:", torch.version.cuda)

GPU: Tesla T4
CUDA: 13.0


In [8]:
%pip install -q "transformers>=4.45,<5" "accelerate>=1.0" "bitsandbytes>=0.45"

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
gradio-client 2.7.2 requires huggingface-hub<3.0,>=1.16.0, but you have huggingface-hub 0.36.2 which is incompatible.
diffusers 0.42.1 requires huggingface-hub<3.0,>=1.32.0, but you have huggingface-hub 0.36.2 which is incompatible.
gradio 6.30.0 requires huggingface-hub<3.0,>=1.16.0, but you have huggingface-hub 0.36.2 which is incompatible.


## 7. Model Initialization

Initialize the instruction-tuned language model and tokenizer using NF4 quantization.

**Model:** Qwen2.5-3B-Instruct  
**Precision:** 4-bit  
**Task:** Structured call summarization

In [9]:
import torch
from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    BitsAndBytesConfig
)

print("=" * 65)
print("STAGE 04 — MODEL INITIALIZATION")
print("=" * 65)

# Model configuration
MODEL_ID = "Qwen/Qwen2.5-3B-Instruct"

quant_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True
)

# Load tokenizer
print("\nLoading tokenizer...")

tokenizer = AutoTokenizer.from_pretrained(
    MODEL_ID
)

# Load model
print("\nLoading model...")

model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    quantization_config=quant_config,
    device_map="auto",
    torch_dtype=torch.float16
)

model.eval()

# Model information
print("\n" + "=" * 65)
print("MODEL INFORMATION")
print("=" * 65)

print(f"Model: {MODEL_ID}")
print(f"Device: {model.device}")
print(f"Quantization: 4-bit NF4")

print(
    f"Model Memory: "
    f"{model.get_memory_footprint() / 1024**3:.2f} GB"
)

print(
    f"GPU Memory Allocated: "
    f"{torch.cuda.memory_allocated() / 1024**3:.2f} GB"
)

print("\nMODEL LOADED SUCCESSFULLY")

STAGE 04 — MODEL INITIALIZATION

Loading tokenizer...


/usr/local/lib/python3.13/dist-packages/huggingface_hub/utils/_auth.py:104: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
You are not authenticated with the Hugging Face Hub in this notebook.
If the error persists, please let us know by opening an issue on GitHub (https://github.com/huggingface/huggingface_hub/issues/new).
  warnings.warn(


tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

... (6 outputs left out)

model-00002-of-00002.safetensors:   0%|          | 0.00/2.20G [00:00<?, ?B/s]

Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]


MODEL INFORMATION
Model: Qwen/Qwen2.5-3B-Instruct
Device: cuda:0
Quantization: 4-bit NF4
Model Memory: 1.87 GB
GPU Memory Allocated: 1.91 GB

MODEL LOADED SUCCESSFULLY


## 8. Initial Summarization Test

Evaluate structured summary generation on CALL_001 before processing the complete dataset.

The model is instructed to extract only transcript-supported information and avoid unsupported conclusions.

In [14]:
import json
import time
from pathlib import Path

print("=" * 65)
print("STAGE 04 — SUMMARIZATION TEST: CALL_001")
print("=" * 65)

# 1. Load prepared transcript
input_path = Path(
    "data/processed/summarization/input/CALL_001.json"
)

with open(input_path, "r", encoding="utf-8") as f:
    call_data = json.load(f)

transcript = call_data["formatted_transcript"]

# 2. Summarization instructions
system_prompt = """
You are a customer service call summarization system.

Analyze the full conversation and produce an accurate,
concise, and structured summary.

Return a valid JSON object with exactly these fields:

{
    "customer_issue": "",
    "agent_action": "",
    "final_outcome": "",
    "summary": "",
    "notes_limitations": ""
}

FIELD DEFINITIONS:

1. customer_issue:
   Identify the customer's main request, problem,
   and important requirements.

2. agent_action:
   Describe the agent's explanations, actions,
   solutions, and commitments.
   Include relevant pricing, service options,
   and delivery details when available.

3. final_outcome:
   State what was actually confirmed or completed.
   Clearly identify any pending or promised actions.

4. summary:
   Provide a concise overview of the customer's request,
   agent response, and final status.

5. notes_limitations:
   Mention unresolved matters, pending commitments,
   or relevant transcript uncertainties.

CRITICAL ACCURACY RULES:

- Use only facts supported by the transcript.
- Never describe promised actions as completed actions.
- Statements such as "I will send", "I'll arrange",
  or "Expect an email" indicate future commitments,
  not completed actions.
- Distinguish service availability from confirmed arrangements.
- Distinguish suggestions from finalized decisions.
- Do not claim an email, document, or confirmation was sent
  unless the conversation explicitly confirms its delivery.
- Preserve important prices, dates, times, and requirements.
- Do not invent missing details.
- Use "Not specified" when information is unavailable.
- Keep each field concise and factual.
- Return valid JSON only.
"""

messages = [
    {
        "role": "system",
        "content": system_prompt
    },
    {
        "role": "user",
        "content": f"Call Transcript:\n\n{transcript}"
    }
]

# 3. Tokenization with attention mask
inputs = tokenizer.apply_chat_template(
    messages,
    tokenize=True,
    add_generation_prompt=True,
    return_dict=True,
    return_tensors="pt"
).to(model.device)

input_tokens = inputs["input_ids"].shape[-1]

print(f"\nCall ID: {call_data['call_id']}")
print(f"Input Tokens: {input_tokens}")

if input_tokens + 400 > 32768:
    raise ValueError("Input exceeds the model context limit.")

# 4. Generate summary
print("\nGenerating summary...")

start_time = time.time()

with torch.inference_mode():

    outputs = model.generate(
        **inputs,
        max_new_tokens=400,
        do_sample=False,
        pad_token_id=tokenizer.eos_token_id
    )

inference_time = time.time() - start_time

# 5. Decode generated text
generated_tokens = outputs[0][input_tokens:]

generated_text = tokenizer.decode(
    generated_tokens,
    skip_special_tokens=True
).strip()

print("\n" + "=" * 65)
print("GENERATED SUMMARY")
print("=" * 65)

# 6. Validate and save JSON
try:
    summary = json.loads(generated_text)

    required_fields = {
        "customer_issue",
        "agent_action",
        "final_outcome",
        "summary",
        "notes_limitations"
    }

    if not isinstance(summary, dict):
        raise ValueError("Generated output is not a JSON object.")

    if not required_fields.issubset(summary):
        raise ValueError("Missing required summary fields.")

    print(json.dumps(
        summary,
        indent=2,
        ensure_ascii=False
    ))

    # Save result
    output_dir = Path(
        "data/processed/summarization/results"
    )

    output_dir.mkdir(parents=True, exist_ok=True)

    result = {
        "call_id": call_data["call_id"],
        "model": MODEL_ID,
        "quantization": "4-bit NF4",
        "input_tokens": input_tokens,
        "inference_seconds": round(inference_time, 2),
        "generated_summary": summary
    }

    output_path = output_dir / "CALL_001_summary.json"

    with open(output_path, "w", encoding="utf-8") as f:
        json.dump(
            result,
            f,
            indent=2,
            ensure_ascii=False
        )

    print(f"\nSaved: {output_path}")

except (json.JSONDecodeError, ValueError) as e:

    print(f"\nJSON validation failed: {e}")
    print("\nRaw output:")
    print(generated_text)

# 7. Inference statistics
print("\n" + "=" * 65)
print("INFERENCE STATISTICS")
print("=" * 65)

print(f"Input Tokens: {input_tokens}")
print(f"Generated Tokens: {len(generated_tokens)}")
print(f"Inference Time: {inference_time:.2f} seconds")

print("\n" + "=" * 65)
print("SUMMARIZATION TEST COMPLETED")
print("=" * 65)

STAGE 04 — SUMMARIZATION TEST: CALL_001

Call ID: CALL_001
Input Tokens: 1820

Generating summary...

GENERATED SUMMARY
{
  "customer_issue": "Corporate catering for a team lunch with dietary restrictions (vegan, gluten-free), inquiry about pricing, menu options, delivery, and theme.",
  "agent_action": "Explained the catering options, confirmed the price range, menu items, and delivery services. Offered to send a sample menu and confirmation email. Confirmed availability for a team lunch on Thursday at noon. Added a theme of Cinco de Mayo to the event.",
  "final_outcome": "Confirmed a team lunch on Thursday at noon. Sent a sample menu and confirmation email with the details. Catering services are arranged with a theme of Cinco de Mayo.",
  "summary": "Customer requested corporate catering for a team lunch with dietary restrictions. The agent provided details about the menu, pricing, delivery, and theme (Cinco de Mayo). A confirmation was sent with the details.",
  "notes_limitations"

## 9. Summary Quality Evaluation

Review the generated summary against the source transcript to assess factual consistency, completeness, and unsupported claims.

In [15]:
import json
from pathlib import Path

print("=" * 65)
print("STAGE 04 — SUMMARY QUALITY EVALUATION")
print("=" * 65)

# Load transcript
with open(
    "data/processed/summarization/input/CALL_001.json",
    "r",
    encoding="utf-8"
) as f:
    call_data = json.load(f)

# Load generated summary
with open(
    "data/processed/summarization/results/CALL_001_summary.json",
    "r",
    encoding="utf-8"
) as f:
    result = json.load(f)

summary = result["generated_summary"]
turns = call_data["speaker_turns"]

# 1. Display generated summary
print("\n[1] GENERATED SUMMARY")
print("-" * 65)

for key, value in summary.items():
    print(f"\n{key.upper()}:")
    print(value)

# 2. Identify supporting transcript evidence
print("\n\n[2] RELEVANT TRANSCRIPT EVIDENCE")
print("-" * 65)

keywords = [
    "thursday",
    "noon",
    "cinco",
    "theme",
    "confirm",
    "menu",
    "dietary",
    "vegan",
    "gluten",
    "delivery",
    "email",
    "send"
]

matched_indices = set()

for i, turn in enumerate(turns):

    text = turn["text"].lower()

    if any(keyword in text for keyword in keywords):

        # Include surrounding context
        for j in range(
            max(0, i - 1),
            min(len(turns), i + 2)
        ):
            matched_indices.add(j)

for i in sorted(matched_indices):

    turn = turns[i]

    print(
        f"\n[{turn['start']:.2f}s] "
        f"{turn['speaker']}: {turn['text']}"
    )

# 3. Review conversation ending
print("\n\n[3] FINAL CONVERSATION TURNS")
print("-" * 65)

for turn in turns[-10:]:

    print(
        f"\n[{turn['start']:.2f}s] "
        f"{turn['speaker']}: {turn['text']}"
    )

print("\n" + "=" * 65)
print("EVIDENCE REVIEW COMPLETED")
print("=" * 65)

STAGE 04 — SUMMARY QUALITY EVALUATION

[1] GENERATED SUMMARY
-----------------------------------------------------------------

CUSTOMER_ISSUE:
Corporate catering for a team lunch with dietary restrictions (vegan, gluten-free), inquiry about pricing, menu options, delivery, and theme.

AGENT_ACTION:
Explained the catering options, confirmed the price range, menu items, and delivery services. Offered to send a sample menu and confirmation email. Confirmed availability for a team lunch on Thursday at noon. Added a theme of Cinco de Mayo to the event.

FINAL_OUTCOME:
Confirmed a team lunch on Thursday at noon. Sent a sample menu and confirmation email with the details. Catering services are arranged with a theme of Cinco de Mayo.

SUMMARY:
Customer requested corporate catering for a team lunch with dietary restrictions. The agent provided details about the menu, pricing, delivery, and theme (Cinco de Mayo). A confirmation was sent with the details.

NOTES_LIMITATIONS:
No email was sent du

## 10. Batch Call Summarization

Generate structured summaries for all 12 calls using Qwen2.5-3B-Instruct.

Validate JSON outputs and record inference metrics for subsequent quality evaluation.

In [16]:
import json
import time
import hashlib
import pandas as pd
import torch
from pathlib import Path

print("=" * 65)
print("STAGE 04 — BATCH CALL SUMMARIZATION")
print("=" * 65)

# 1. Directories
INPUT_DIR = Path("data/processed/summarization/input")
RESULTS_DIR = Path("data/processed/summarization/results")

RESULTS_DIR.mkdir(parents=True, exist_ok=True)

# 2. Load selected calls
selected_calls = pd.read_csv(
    "data/processed/selected_calls.csv"
)

call_ids = selected_calls["call_id"].tolist()

# 3. Generation settings
MAX_NEW_TOKENS = 500
MAX_CONTEXT_TOKENS = 32768

required_fields = {
    "customer_issue",
    "agent_action",
    "final_outcome",
    "summary",
    "notes_limitations"
}

# Save prompt for reproducibility
prompt_hash = hashlib.sha256(
    system_prompt.encode("utf-8")
).hexdigest()

with open(
    RESULTS_DIR / "system_prompt.txt",
    "w",
    encoding="utf-8"
) as f:
    f.write(system_prompt)

print(f"\nModel: {MODEL_ID}")
print(f"Total Calls: {len(call_ids)}")
print(f"Prompt SHA256: {prompt_hash[:16]}...")

# 4. Batch inference
batch_results = []

for index, call_id in enumerate(call_ids, start=1):

    print("\n" + "-" * 65)
    print(f"[{index}/{len(call_ids)}] Processing {call_id}")
    print("-" * 65)

    input_path = INPUT_DIR / f"{call_id}.json"
    output_path = RESULTS_DIR / f"{call_id}_summary.json"

    # Skip previously processed calls
    if output_path.exists():

        print("Already processed — skipped.")

        batch_results.append({
            "call_id": call_id,
            "status": "skipped_existing",
            "input_tokens": None,
            "generated_tokens": None,
            "inference_seconds": None
        })

        continue

    try:

        # Load transcript
        with open(input_path, "r", encoding="utf-8") as f:
            call_data = json.load(f)

        transcript = call_data["formatted_transcript"]

        messages = [
            {
                "role": "system",
                "content": system_prompt
            },
            {
                "role": "user",
                "content": f"Call Transcript:\n\n{transcript}"
            }
        ]

        # Tokenization
        inputs = tokenizer.apply_chat_template(
            messages,
            tokenize=True,
            add_generation_prompt=True,
            return_dict=True,
            return_tensors="pt"
        ).to(model.device)

        input_tokens = inputs["input_ids"].shape[-1]

        if input_tokens + MAX_NEW_TOKENS > MAX_CONTEXT_TOKENS:
            raise ValueError("Context length exceeded.")

        # Generate summary
        start_time = time.time()

        with torch.inference_mode():

            outputs = model.generate(
                **inputs,
                max_new_tokens=MAX_NEW_TOKENS,
                do_sample=False,
                pad_token_id=tokenizer.eos_token_id
            )

        inference_time = time.time() - start_time

        generated_tokens = outputs[0][input_tokens:]

        generated_text = tokenizer.decode(
            generated_tokens,
            skip_special_tokens=True
        ).strip()

        # Validate JSON
        try:

            summary = json.loads(generated_text)

            if not isinstance(summary, dict):
                raise ValueError("Output is not a JSON object.")

            if set(summary.keys()) != required_fields:
                raise ValueError("Unexpected or missing JSON fields.")

            if not all(
                isinstance(value, str)
                for value in summary.values()
            ):
                raise ValueError("Summary fields must be strings.")

        except (json.JSONDecodeError, ValueError):

            raw_path = RESULTS_DIR / f"{call_id}_raw.txt"

            raw_path.write_text(
                generated_text,
                encoding="utf-8"
            )

            raise ValueError(
                f"Invalid JSON. Raw output saved: {raw_path}"
            )

        # Save result
        result = {
            "call_id": call_id,
            "model": MODEL_ID,
            "quantization": "4-bit NF4",
            "prompt_sha256": prompt_hash,
            "input_tokens": input_tokens,
            "generated_tokens": len(generated_tokens),
            "inference_seconds": round(inference_time, 2),
            "generated_summary": summary
        }

        with open(
            output_path,
            "w",
            encoding="utf-8"
        ) as f:
            json.dump(
                result,
                f,
                indent=2,
                ensure_ascii=False
            )

        status = "success"

        print(f"Status: {status}")
        print(f"Input Tokens: {input_tokens}")
        print(f"Generated Tokens: {len(generated_tokens)}")
        print(f"Inference Time: {inference_time:.2f}s")

        batch_results.append({
            "call_id": call_id,
            "status": status,
            "input_tokens": input_tokens,
            "generated_tokens": len(generated_tokens),
            "inference_seconds": round(inference_time, 2)
        })

    except Exception as e:

        print(f"ERROR: {e}")

        batch_results.append({
            "call_id": call_id,
            "status": "failed",
            "input_tokens": None,
            "generated_tokens": None,
            "inference_seconds": None
        })

# 5. Save batch statistics
batch_df = pd.DataFrame(batch_results)

batch_df.to_csv(
    RESULTS_DIR / "batch_inference_metrics.csv",
    index=False
)

# 6. Display summary
print("\n" + "=" * 65)
print("BATCH SUMMARIZATION RESULTS")
print("=" * 65)

display(batch_df)

print("\n" + "=" * 65)
print("PROCESSING SUMMARY")
print("=" * 65)

print(f"Total Calls: {len(batch_df)}")

print(
    f"Successfully Generated: "
    f"{(batch_df['status'] == 'success').sum()}"
)

print(
    f"Previously Processed: "
    f"{(batch_df['status'] == 'skipped_existing').sum()}"
)

print(
    f"Failed: "
    f"{(batch_df['status'] == 'failed').sum()}"
)

saved_summaries = list(
    RESULTS_DIR.glob("CALL_*_summary.json")
)

print(f"Total Saved Summaries: {len(saved_summaries)}")

print("\nBATCH SUMMARIZATION COMPLETED")

STAGE 04 — BATCH CALL SUMMARIZATION

Model: Qwen/Qwen2.5-3B-Instruct
Total Calls: 12
Prompt SHA256: 9b7478816a859d7f...

-----------------------------------------------------------------
[1/12] Processing CALL_001
-----------------------------------------------------------------
Already processed — skipped.

-----------------------------------------------------------------
[2/12] Processing CALL_002
-----------------------------------------------------------------
Status: success
Input Tokens: 2920
Generated Tokens: 332
Inference Time: 25.17s

-----------------------------------------------------------------
[3/12] Processing CALL_003
-----------------------------------------------------------------
Status: success
Input Tokens: 3715
Generated Tokens: 235
Inference Time: 20.64s

-----------------------------------------------------------------
[4/12] Processing CALL_004
-----------------------------------------------------------------
Status: success
Input Tokens: 2657
Generated Tokens

,call_id,status,input_tokens,generated_tokens,inference_seconds
0,CALL_001,skipped_existing,NaN,NaN,NaN
1,CALL_002,success,2920.0,332.0,25.17
2,CALL_003,success,3715.0,235.0,20.64
3,CALL_004,success,2657.0,259.0,19.59
4,CALL_005,success,3875.0,302.0,25.57
5,CALL_006,success,3085.0,244.0,19.52
6,CALL_007,success,3849.0,285.0,24.78
7,CALL_008,success,3437.0,391.0,29.90
8,CALL_009,success,2652.0,240.0,18.55
9,CALL_010,success,2409.0,259.0,19.40



PROCESSING SUMMARY
Total Calls: 12
Successfully Generated: 10
Previously Processed: 1
Failed: 1
Total Saved Summaries: 11

BATCH SUMMARIZATION COMPLETED


## 11. Failed Generation Analysis

Inspect the invalid JSON output from CALL_012 to identify formatting errors or incomplete generation.

In [17]:
import json
from pathlib import Path

print("=" * 65)
print("STAGE 04 — FAILED GENERATION ANALYSIS")
print("=" * 65)

# 1. Load raw model output
raw_path = Path(
    "data/processed/summarization/results/CALL_012_raw.txt"
)

raw_text = raw_path.read_text(encoding="utf-8")

# 2. Output statistics
print("\n[1] RAW OUTPUT STATISTICS")
print("-" * 65)

print(f"Characters: {len(raw_text)}")
print(f"Words: {len(raw_text.split())}")

generated_token_count = len(
    tokenizer.encode(
        raw_text,
        add_special_tokens=False
    )
)

print(f"Approximate Generated Tokens: {generated_token_count}")
print(f"Generation Limit: 500")

# 3. Inspect output
print("\n[2] RAW MODEL OUTPUT")
print("-" * 65)

print(raw_text)

# 4. Diagnose JSON
print("\n[3] JSON VALIDATION")
print("-" * 65)

try:
    parsed = json.loads(raw_text)

    print("JSON Syntax: Valid")
    print("Output Type:", type(parsed).__name__)

except json.JSONDecodeError as e:

    print("JSON Syntax: Invalid")
    print(f"Error: {e.msg}")
    print(f"Line: {e.lineno}")
    print(f"Column: {e.colno}")
    print(f"Position: {e.pos}")

    print("\nText near error:")

    start = max(0, e.pos - 120)
    end = min(len(raw_text), e.pos + 120)

    print(repr(raw_text[start:end]))

# 5. Check possible truncation
print("\n[4] TRUNCATION CHECK")
print("-" * 65)

if generated_token_count >= 495:
    print("Possible truncation: Output reached token limit.")
else:
    print("Output appears below token limit.")

print("\n" + "=" * 65)
print("ANALYSIS COMPLETED")
print("=" * 65)

STAGE 04 — FAILED GENERATION ANALYSIS

[1] RAW OUTPUT STATISTICS
-----------------------------------------------------------------
Characters: 1704
Words: 274
Approximate Generated Tokens: 369
Generation Limit: 500

[2] RAW MODEL OUTPUT
-----------------------------------------------------------------
```json
{
    "customer_issue": "The customer wants to plan a brunch for 20 people with specific dietary requirements and preferences. The customer is seeking a quote and wants to include a barista for coffee.",
    "agent_action": "The agent asked for details about the venue, dietary requirements, and other preferences. The agent offered various menu items and explained the process for creating a custom menu. The agent confirmed the vegetarian and vegan options and the inclusion of a barista. The agent also provided information about the coffee machine and smoothies. The agent scheduled the delivery for 9:30 AM and included a barista for 3 hours. The agent also added cleanup services and

## 12. JSON Format Recovery

Remove Markdown formatting from CALL_012 output, validate the JSON structure, and recover the generated summary without rerunning inference.

In [18]:
import json
import re
import hashlib
import pandas as pd
from pathlib import Path

print("=" * 65)
print("STAGE 04 — JSON FORMAT RECOVERY")
print("=" * 65)

RESULTS_DIR = Path("data/processed/summarization/results")

raw_path = RESULTS_DIR / "CALL_012_raw.txt"
output_path = RESULTS_DIR / "CALL_012_summary.json"

# 1. Load original model output
raw_text = raw_path.read_text(encoding="utf-8").strip()

# 2. Remove Markdown code fences
match = re.fullmatch(
    r"```(?:json)?[ \t]*\r?\n(.*?)\r?\n```",
    raw_text,
    flags=re.DOTALL | re.IGNORECASE
)

cleaned_text = match.group(1) if match else raw_text

# 3. Validate JSON
summary = json.loads(cleaned_text)

required_fields = {
    "customer_issue",
    "agent_action",
    "final_outcome",
    "summary",
    "notes_limitations"
}

if not isinstance(summary, dict):
    raise ValueError("Output must be a JSON object.")

if set(summary.keys()) != required_fields:
    raise ValueError("Missing or unexpected JSON fields.")

if not all(isinstance(v, str) for v in summary.values()):
    raise ValueError("All summary fields must be strings.")

print("\nJSON Validation: PASSED")

# 4. Retrieve prompt metadata
prompt_path = RESULTS_DIR / "system_prompt.txt"

saved_prompt = prompt_path.read_text(encoding="utf-8")

prompt_hash = hashlib.sha256(
    saved_prompt.encode("utf-8")
).hexdigest()

# 5. Save recovered result
result = {
    "call_id": "CALL_012",
    "model": "Qwen/Qwen2.5-3B-Instruct",
    "quantization": "4-bit NF4",
    "prompt_sha256": prompt_hash,
    "inference_seconds": None,
    "recovery_method": "Markdown code fence removal",
    "source_raw_output": raw_path.name,
    "generated_summary": summary
}

with open(output_path, "w", encoding="utf-8") as f:
    json.dump(result, f, indent=2, ensure_ascii=False)

print(f"Saved: {output_path}")

# 6. Update batch metrics
metrics_path = RESULTS_DIR / "batch_inference_metrics.csv"

metrics_df = pd.read_csv(metrics_path)

metrics_df.loc[
    metrics_df["call_id"] == "CALL_012",
    "status"
] = "format_recovered"

metrics_df.to_csv(metrics_path, index=False)

# 7. Verify all summaries
saved_summaries = sorted(
    RESULTS_DIR.glob("CALL_*_summary.json")
)

print("\n" + "=" * 65)
print("FINAL SUMMARIZATION STATUS")
print("=" * 65)

print(f"Total Saved Summaries: {len(saved_summaries)}")
print(f"Expected Summaries: 12")

for file in saved_summaries:
    print(f"  {file.name}")

print("\n" + "=" * 65)

if len(saved_summaries) == 12:
    print("SUCCESS: All 12 call summaries are available.")
else:
    print("WARNING: Some summaries are still missing.")

print("=" * 65)

STAGE 04 — JSON FORMAT RECOVERY

JSON Validation: PASSED
Saved: data/processed/summarization/results/CALL_012_summary.json

FINAL SUMMARIZATION STATUS
Total Saved Summaries: 12
Expected Summaries: 12
  CALL_001_summary.json
  CALL_002_summary.json
  CALL_003_summary.json
  CALL_004_summary.json
  CALL_005_summary.json
  CALL_006_summary.json
  CALL_007_summary.json
  CALL_008_summary.json
  CALL_009_summary.json
  CALL_010_summary.json
  CALL_011_summary.json
  CALL_012_summary.json

SUCCESS: All 12 call summaries are available.


## 13. Summary Quality Audit

Prepare the generated summaries for factual accuracy and completeness evaluation.

Review key details, completed actions, pending commitments, and potential contradictions against the source transcripts.

In [19]:
import json
import re
import pandas as pd
from pathlib import Path

print("=" * 65)
print("STAGE 04 — SUMMARY QUALITY AUDIT")
print("=" * 65)

# 1. Define paths
INPUT_DIR = Path("data/processed/summarization/input")
RESULTS_DIR = Path("data/processed/summarization/results")
EVAL_DIR = Path("data/processed/summarization/evaluation")

EVAL_DIR.mkdir(parents=True, exist_ok=True)

selected_calls = pd.read_csv(
    "data/processed/selected_calls.csv"
)

required_fields = [
    "customer_issue",
    "agent_action",
    "final_outcome",
    "summary",
    "notes_limitations"
]

audit_records = []
review_packets = [
    "# Call Summarization — Quality Review\n",
    "Generated summaries and source transcripts for manual evaluation.\n"
]

# 2. Inspect all summaries
for call_id in selected_calls["call_id"]:

    transcript_path = INPUT_DIR / f"{call_id}.json"
    summary_path = RESULTS_DIR / f"{call_id}_summary.json"

    with open(transcript_path, "r", encoding="utf-8") as f:
        call_data = json.load(f)

    with open(summary_path, "r", encoding="utf-8") as f:
        result = json.load(f)

    summary = result["generated_summary"]
    transcript = call_data["formatted_transcript"]

    # Validate structure
    structure_valid = (
        isinstance(summary, dict)
        and set(summary.keys()) == set(required_fields)
        and all(
            isinstance(summary.get(field), str)
            and bool(summary[field].strip())
            for field in required_fields
        )
    )

    # Identify potentially sensitive claims
    combined_summary = " ".join(
        str(summary.get(field, ""))
        for field in required_fields
    ).lower()

    review_flags = []

    if re.search(
        r"\b(sent|emailed|delivered|refunded|paid)\b",
        combined_summary
    ):
        review_flags.append("Completed action")

    if re.search(
        r"\b(confirmed|booked|scheduled|resolved)\b",
        combined_summary
    ):
        review_flags.append("Confirmed outcome")

    if re.search(
        r"\b(pending|promised|will send|not sent|not yet)\b",
        combined_summary
    ):
        review_flags.append("Pending commitment")

    if (
        "Completed action" in review_flags
        and "Pending commitment" in review_flags
    ):
        review_flags.append("Check action consistency")

    # Create audit record
    audit_records.append({
        "call_id": call_id,
        "domain": call_data["domain"],
        "structure_valid": structure_valid,
        "review_flags": ", ".join(review_flags),
        "customer_issue": summary.get("customer_issue", ""),
        "agent_action": summary.get("agent_action", ""),
        "final_outcome": summary.get("final_outcome", ""),
        "summary": summary.get("summary", ""),
        "notes_limitations": summary.get("notes_limitations", ""),
        "factual_accuracy_1_5": None,
        "completeness_1_5": None,
        "outcome_accuracy_1_5": None,
        "review_notes": "",
        "review_status": "pending"
    })

    # Create full review document
    review_packets.extend([
        f"\n---\n## {call_id}\n",
        "### Generated Summary\n",
        "```json",
        json.dumps(summary, indent=2, ensure_ascii=False),
        "```",
        "\n### Source Transcript\n",
        "```text",
        transcript,
        "```"
    ])

# 3. Save evaluation files
audit_df = pd.DataFrame(audit_records)

audit_path = EVAL_DIR / "summary_quality_audit.csv"

audit_df.to_csv(
    audit_path,
    index=False
)

review_path = EVAL_DIR / "summary_review_packets.md"

review_path.write_text(
    "\n".join(review_packets),
    encoding="utf-8"
)

# 4. Display results
print("\nSUMMARY VALIDATION")
print("-" * 65)

print(f"Total Calls: {len(audit_df)}")

print(
    f"Valid Structures: "
    f"{audit_df['structure_valid'].sum()}"
)

print(
    f"Invalid Structures: "
    f"{(~audit_df['structure_valid']).sum()}"
)

print("\nREVIEW OVERVIEW")
print("-" * 65)

display(
    audit_df[
        [
            "call_id",
            "domain",
            "structure_valid",
            "review_flags",
            "review_status"
        ]
    ]
)

print("\nOUTPUT FILES")
print("-" * 65)

print(f"Audit CSV: {audit_path}")
print(f"Review Packets: {review_path}")

print("\n" + "=" * 65)
print("QUALITY AUDIT PREPARATION COMPLETED")
print("=" * 65)

STAGE 04 — SUMMARY QUALITY AUDIT

SUMMARY VALIDATION
-----------------------------------------------------------------
Total Calls: 12
Valid Structures: 12
Invalid Structures: 0

REVIEW OVERVIEW
-----------------------------------------------------------------


,call_id,domain,structure_valid,review_flags,review_status
0,CALL_001,food,True,"Completed action, Confirmed outcome",pending
1,CALL_002,hospitality,True,"Completed action, Confirmed outcome",pending
2,CALL_003,entertainment,True,Confirmed outcome,pending
3,CALL_004,hospitality,True,Confirmed outcome,pending
4,CALL_005,retail,True,"Completed action, Confirmed outcome",pending
5,CALL_006,entertainment,True,"Confirmed outcome, Pending commitment",pending
6,CALL_007,telecom,True,"Completed action, Confirmed outcome",pending
7,CALL_008,insurance,True,Confirmed outcome,pending
8,CALL_009,health,True,Confirmed outcome,pending
9,CALL_010,banking,True,"Completed action, Confirmed outcome",pending



OUTPUT FILES
-----------------------------------------------------------------
Audit CSV: data/processed/summarization/evaluation/summary_quality_audit.csv
Review Packets: data/processed/summarization/evaluation/summary_review_packets.md

QUALITY AUDIT PREPARATION COMPLETED


## 14. Summary Quality Review

Evaluate generated summaries against source transcripts using three criteria:

- Factual Accuracy
- Completeness
- Outcome Accuracy

Record evaluation scores and identify unsupported or missing information.

In [20]:
from google.colab import files
from pathlib import Path

print("=" * 65)
print("STAGE 04 — EXPORT QUALITY REVIEW DATA")
print("=" * 65)

EVAL_DIR = Path(
    "data/processed/summarization/evaluation"
)

review_path = EVAL_DIR / "summary_review_packets.md"
audit_path = EVAL_DIR / "summary_quality_audit.csv"

for path in [review_path, audit_path]:
    if not path.exists():
        raise FileNotFoundError(f"Missing: {path}")

print("\nReview files verified:")
print(f"1. {review_path.name}")
print(f"2. {audit_path.name}")

print("\nDownloading review document...")

files.download(str(review_path))

print("\nExport completed.")

STAGE 04 — EXPORT QUALITY REVIEW DATA

Review files verified:
1. summary_review_packets.md
2. summary_quality_audit.csv



<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>


Export completed.


## 15. Summary Evaluation Results

Evaluate the generated summaries against their source transcripts using factual accuracy, completeness, and outcome accuracy.

Scores range from 1 (poor) to 5 (excellent).

In [22]:
import pandas as pd
import io
from pathlib import Path
from google.colab import files

print("=" * 65)
print("STAGE 04 — SUMMARY EVALUATION RESULTS")
print("=" * 65)

# 1. Upload evaluation file
uploaded = files.upload()

filename = next(
    name for name in uploaded
    if name.startswith("summary_quality_review_scored")
    and name.endswith(".csv")
)

review_df = pd.read_csv(
    io.BytesIO(uploaded[filename])
)

# 2. Load existing audit
EVAL_DIR = Path("data/processed/summarization/evaluation")
audit_path = EVAL_DIR / "summary_quality_audit.csv"

audit_df = pd.read_csv(audit_path)

# 3. Validate call IDs
assert review_df["call_id"].is_unique
assert audit_df["call_id"].is_unique

assert set(review_df["call_id"]) == set(audit_df["call_id"])

# 4. Merge evaluation results
review_columns = [
    col for col in review_df.columns
    if col != "call_id"
]

audit_df = audit_df.drop(
    columns=review_columns,
    errors="ignore"
)

audit_df = audit_df.merge(
    review_df,
    on="call_id",
    how="left",
    validate="one_to_one"
)

# 5. Save updated evaluation
audit_df.to_csv(
    audit_path,
    index=False
)

# 6. Display evaluation results
score_columns = [
    "factual_accuracy_1_5",
    "completeness_1_5",
    "outcome_accuracy_1_5"
]

print("\nEVALUATION SCORES")
print("-" * 65)

display(
    audit_df[["call_id"] + score_columns]
)

print("\n" + "=" * 65)
print("AVERAGE EVALUATION SCORES")
print("=" * 65)

for column in score_columns:
    print(f"{column}: {audit_df[column].mean():.2f}/5")

overall_score = audit_df[score_columns].mean().mean()

print(f"\nOverall Quality Score: {overall_score:.2f}/5")

print(f"\nEvaluated Calls: {len(audit_df)}")
print(f"Saved: {audit_path}")

print("\nSUMMARY EVALUATION COMPLETED")

STAGE 04 — SUMMARY EVALUATION RESULTS


Saving summary_quality_review_scored.csv to summary_quality_review_scored.csv

EVALUATION SCORES
-----------------------------------------------------------------


,call_id,factual_accuracy_1_5,completeness_1_5,outcome_accuracy_1_5
0,CALL_001,3,4,2
1,CALL_002,5,5,5
2,CALL_003,3,4,4
3,CALL_004,2,3,1
4,CALL_005,3,3,3
5,CALL_006,2,3,2
6,CALL_007,3,4,5
7,CALL_008,2,3,1
8,CALL_009,4,4,5
9,CALL_010,2,3,2



AVERAGE EVALUATION SCORES
factual_accuracy_1_5: 3.08/5
completeness_1_5: 3.67/5
outcome_accuracy_1_5: 3.17/5

Overall Quality Score: 3.31/5

Evaluated Calls: 12
Saved: data/processed/summarization/evaluation/summary_quality_audit.csv

SUMMARY EVALUATION COMPLETED


## 16. Export Summarization Results

Export transcripts, generated summaries, inference metrics, and evaluation results for reproducibility.

In [23]:
import zipfile
from pathlib import Path
from google.colab import files

print("=" * 65)
print("STAGE 04 — EXPORT SUMMARIZATION RESULTS")
print("=" * 65)

SOURCE_DIR = Path("data/processed/summarization")

ZIP_PATH = Path("Maharah_Summarization_Results.zip")

if not SOURCE_DIR.exists():
    raise FileNotFoundError("Summarization directory not found.")

# Create ZIP archive
with zipfile.ZipFile(
    ZIP_PATH,
    "w",
    zipfile.ZIP_DEFLATED
) as zipf:

    for file_path in SOURCE_DIR.rglob("*"):

        if file_path.is_file():

            zipf.write(
                file_path,
                arcname=str(file_path)
            )

# Verify archive
with zipfile.ZipFile(ZIP_PATH, "r") as zipf:

    archived_files = zipf.namelist()
    corrupted_file = zipf.testzip()

    if corrupted_file is not None:
        raise RuntimeError(
            f"Corrupted file: {corrupted_file}"
        )

print(f"\nArchive: {ZIP_PATH}")
print(f"Total Files: {len(archived_files)}")
print(f"Size: {ZIP_PATH.stat().st_size / 1024**2:.2f} MB")

print("\nArchive verified successfully.")

# Download
files.download(str(ZIP_PATH))

print("\nEXPORT COMPLETED")

STAGE 04 — EXPORT SUMMARIZATION RESULTS

Archive: Maharah_Summarization_Results.zip
Total Files: 31
Size: 0.16 MB

Archive verified successfully.


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>


EXPORT COMPLETED


## 17. Model Comparison — Qwen3-4B

Evaluate Qwen3-4B-Instruct-2507 on four challenging calls using the same summarization prompt.

Compare factual accuracy, completeness, and outcome accuracy against the Qwen2.5-3B baseline.

In [24]:
import gc
import torch
import transformers

from packaging.version import Version
from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    BitsAndBytesConfig
)

print("=" * 65)
print("STAGE 04 — QWEN3 MODEL COMPARISON SETUP")
print("=" * 65)

# 1. Check environment
print("\n[1] ENVIRONMENT")
print("-" * 65)

print(f"Transformers Version: {transformers.__version__}")
print(f"CUDA Available: {torch.cuda.is_available()}")

if not torch.cuda.is_available():
    raise RuntimeError("GPU is required for this experiment.")

if Version(transformers.__version__) < Version("4.51.0"):
    raise RuntimeError(
        "Qwen3 requires transformers >= 4.51.0."
    )

print(f"GPU: {torch.cuda.get_device_name(0)}")

# 2. Release previous model
print("\n[2] RELEASING PREVIOUS MODEL")
print("-" * 65)

if "model" in globals():
    del model

if "tokenizer" in globals():
    del tokenizer

gc.collect()
torch.cuda.empty_cache()

print(
    f"GPU Memory Allocated: "
    f"{torch.cuda.memory_allocated() / 1024**3:.2f} GB"
)

# 3. Configure new model
print("\n[3] QWEN3 CONFIGURATION")
print("-" * 65)

MODEL_Q3_ID = "Qwen/Qwen3-4B-Instruct-2507"

quant_config_q3 = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True
)

# 4. Load tokenizer
print("\nLoading Qwen3 tokenizer...")

tokenizer_q3 = AutoTokenizer.from_pretrained(
    MODEL_Q3_ID
)

# 5. Load model
print("\nLoading Qwen3 model...")

model_q3 = AutoModelForCausalLM.from_pretrained(
    MODEL_Q3_ID,
    quantization_config=quant_config_q3,
    device_map="auto",
    torch_dtype=torch.float16
)

model_q3.eval()

# 6. Model information
print("\n" + "=" * 65)
print("QWEN3 MODEL INFORMATION")
print("=" * 65)

print(f"Model: {MODEL_Q3_ID}")
print(f"Device: {model_q3.device}")
print("Quantization: 4-bit NF4")

print(
    f"Model Memory: "
    f"{model_q3.get_memory_footprint() / 1024**3:.2f} GB"
)

print(
    f"GPU Memory Allocated: "
    f"{torch.cuda.memory_allocated() / 1024**3:.2f} GB"
)

print("\nQWEN3 MODEL LOADED SUCCESSFULLY")

STAGE 04 — QWEN3 MODEL COMPARISON SETUP

[1] ENVIRONMENT
-----------------------------------------------------------------
Transformers Version: 4.57.6
CUDA Available: True
GPU: Tesla T4

[2] RELEASING PREVIOUS MODEL
-----------------------------------------------------------------
GPU Memory Allocated: 0.01 GB

[3] QWEN3 CONFIGURATION
-----------------------------------------------------------------

Loading Qwen3 tokenizer...


tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json:   0%|          | 0.00/11.4M [00:00<?, ?B/s]


Loading Qwen3 model...


... (5 outputs left out)

model-00002-of-00003.safetensors:   0%|          | 0.00/3.99G [00:00<?, ?B/s]

Loading checkpoint shards:   0%|          | 0/3 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/238 [00:00<?, ?B/s]


QWEN3 MODEL INFORMATION
Model: Qwen/Qwen3-4B-Instruct-2507
Device: cuda:0
Quantization: 4-bit NF4
Model Memory: 2.42 GB
GPU Memory Allocated: 2.49 GB

QWEN3 MODEL LOADED SUCCESSFULLY


## 18. Qwen3 Summarization Experiment

Generate structured summaries for four challenging calls using Qwen3-4B-Instruct-2507.

Use the same transcripts and prompt as the baseline model to compare summary quality and factual consistency.

In [25]:
import json
import time
import re
import hashlib
import pandas as pd
import torch
from pathlib import Path

print("=" * 65)
print("STAGE 04 — QWEN3 SUMMARIZATION EXPERIMENT")
print("=" * 65)

# 1. Configuration
TEST_CALLS = [
    "CALL_001",
    "CALL_004",
    "CALL_008",
    "CALL_010"
]

INPUT_DIR = Path(
    "data/processed/summarization/input"
)

BASELINE_DIR = Path(
    "data/processed/summarization/results"
)

Q3_OUTPUT_DIR = Path(
    "data/processed/summarization/model_comparison/qwen3_4b_2507"
)

Q3_OUTPUT_DIR.mkdir(
    parents=True,
    exist_ok=True
)

MAX_NEW_TOKENS = 500

# 2. Load original prompt
prompt_path = BASELINE_DIR / "system_prompt.txt"

comparison_prompt = prompt_path.read_text(
    encoding="utf-8"
)

prompt_hash = hashlib.sha256(
    comparison_prompt.encode("utf-8")
).hexdigest()

required_fields = {
    "customer_issue",
    "agent_action",
    "final_outcome",
    "summary",
    "notes_limitations"
}

print(f"\nModel: {MODEL_Q3_ID}")
print(f"Test Calls: {len(TEST_CALLS)}")
print(f"Prompt SHA256: {prompt_hash[:16]}...")
print(f"Max New Tokens: {MAX_NEW_TOKENS}")

# 3. Verify baseline prompt consistency
for call_id in TEST_CALLS:

    baseline_path = (
        BASELINE_DIR / f"{call_id}_summary.json"
    )

    with open(baseline_path, "r", encoding="utf-8") as f:
        baseline = json.load(f)

    baseline_hash = baseline.get("prompt_sha256")

    if baseline_hash and baseline_hash != prompt_hash:
        raise ValueError(
            f"Prompt mismatch detected for {call_id}"
        )

print("\nBaseline prompt verification: PASSED")

# 4. Run inference
experiment_results = []

for index, call_id in enumerate(TEST_CALLS, start=1):

    print("\n" + "=" * 65)
    print(f"[{index}/{len(TEST_CALLS)}] {call_id}")
    print("=" * 65)

    input_path = INPUT_DIR / f"{call_id}.json"

    output_path = (
        Q3_OUTPUT_DIR / f"{call_id}_summary.json"
    )

    with open(input_path, "r", encoding="utf-8") as f:
        call_data = json.load(f)

    transcript = call_data["formatted_transcript"]

    messages = [
        {
            "role": "system",
            "content": comparison_prompt
        },
        {
            "role": "user",
            "content": f"Call Transcript:\n\n{transcript}"
        }
    ]

    # Tokenization
    inputs = tokenizer_q3.apply_chat_template(
        messages,
        tokenize=True,
        add_generation_prompt=True,
        return_dict=True,
        return_tensors="pt"
    ).to(model_q3.device)

    input_tokens = inputs["input_ids"].shape[-1]

    print(f"\nInput Tokens: {input_tokens}")

    if input_tokens + MAX_NEW_TOKENS > 16000:
        raise ValueError(
            f"{call_id}: Input exceeds experiment limit."
        )

    # Generate summary
    print("Generating summary...")

    torch.cuda.synchronize()
    start_time = time.perf_counter()

    with torch.inference_mode():

        outputs = model_q3.generate(
            **inputs,
            max_new_tokens=MAX_NEW_TOKENS,
            do_sample=False,
            pad_token_id=tokenizer_q3.eos_token_id
        )

    torch.cuda.synchronize()
    inference_time = time.perf_counter() - start_time

    generated_tokens = outputs[0][input_tokens:]

    generated_text = tokenizer_q3.decode(
        generated_tokens,
        skip_special_tokens=True
    ).strip()

    # 5. JSON recovery
    raw_text = generated_text

    match = re.fullmatch(
        r"```(?:json)?[ \t]*\r?\n(.*?)\r?\n```",
        generated_text,
        flags=re.DOTALL | re.IGNORECASE
    )

    format_recovered = match is not None

    if format_recovered:
        generated_text = match.group(1).strip()

    try:

        summary = json.loads(generated_text)

        if not isinstance(summary, dict):
            raise ValueError("Output is not a JSON object.")

        if set(summary.keys()) != required_fields:
            raise ValueError("Missing or unexpected fields.")

        if not all(
            isinstance(value, str)
            for value in summary.values()
        ):
            raise ValueError("Invalid field types.")

        status = "success"

        # Save result
        result = {
            "call_id": call_id,
            "model": MODEL_Q3_ID,
            "quantization": "4-bit NF4",
            "prompt_sha256": prompt_hash,
            "input_tokens": input_tokens,
            "generated_tokens": len(generated_tokens),
            "inference_seconds": round(inference_time, 2),
            "format_recovered": format_recovered,
            "generated_summary": summary
        }

        with open(
            output_path,
            "w",
            encoding="utf-8"
        ) as f:

            json.dump(
                result,
                f,
                indent=2,
                ensure_ascii=False
            )

        print("\nGENERATED SUMMARY")
        print("-" * 65)

        print(json.dumps(
            summary,
            indent=2,
            ensure_ascii=False
        ))

    except (json.JSONDecodeError, ValueError) as e:

        status = "failed"

        raw_path = (
            Q3_OUTPUT_DIR / f"{call_id}_raw.txt"
        )

        raw_path.write_text(
            raw_text,
            encoding="utf-8"
        )

        print(f"\nJSON Validation Failed: {e}")
        print(f"Raw output saved: {raw_path}")

    # 6. Record statistics
    experiment_results.append({
        "call_id": call_id,
        "model": MODEL_Q3_ID,
        "status": status,
        "input_tokens": input_tokens,
        "generated_tokens": len(generated_tokens),
        "inference_seconds": round(inference_time, 2),
        "format_recovered": format_recovered
    })

    print(f"\nStatus: {status}")
    print(f"Inference Time: {inference_time:.2f}s")

# 7. Save experiment statistics
experiment_df = pd.DataFrame(
    experiment_results
)

metrics_path = (
    Q3_OUTPUT_DIR / "experiment_metrics.csv"
)

experiment_df.to_csv(
    metrics_path,
    index=False
)

# 8. Display results
print("\n" + "=" * 65)
print("QWEN3 EXPERIMENT RESULTS")
print("=" * 65)

display(experiment_df)

print("\n" + "=" * 65)
print("EXPERIMENT SUMMARY")
print("=" * 65)

print(f"Total Calls: {len(experiment_df)}")

print(
    f"Successful: "
    f"{(experiment_df['status'] == 'success').sum()}"
)

print(
    f"Failed: "
    f"{(experiment_df['status'] == 'failed').sum()}"
)

print(
    f"Average Inference Time: "
    f"{experiment_df['inference_seconds'].mean():.2f}s"
)

print(f"\nResults Directory: {Q3_OUTPUT_DIR}")

print("\nQWEN3 EXPERIMENT COMPLETED")

STAGE 04 — QWEN3 SUMMARIZATION EXPERIMENT

Model: Qwen/Qwen3-4B-Instruct-2507
Test Calls: 4
Prompt SHA256: 9b7478816a859d7f...
Max New Tokens: 500

Baseline prompt verification: PASSED

[1/4] CALL_001

Input Tokens: 1820
Generating summary...

GENERATED SUMMARY
-----------------------------------------------------------------
{
  "customer_issue": "Ashley is looking into corporate catering for a team lunch with dietary restrictions (vegan and gluten-free), wants to know pricing, menu options, delivery setup, and if last-minute changes (like keto or vegan) can be accommodated. She also wants a sample menu sent to her email and a theme for the event.",
  "agent_action": "The agent confirmed the corporate lunch package starts at $25 per person, including main, side, and sweet treat. Menu options include quinoa salads, spicy jackfruit tacos, and chocolate avocado moose. Delivery and setup were confirmed with full service including napkins, plates, and styling. The agent agreed to send a sa

,call_id,model,status,input_tokens,generated_tokens,inference_seconds,format_recovered
0,CALL_001,Qwen/Qwen3-4B-Instruct-2507,success,1820,427,39.34,False
1,CALL_004,Qwen/Qwen3-4B-Instruct-2507,success,2657,478,50.85,False
2,CALL_008,Qwen/Qwen3-4B-Instruct-2507,failed,3437,500,65.68,False
3,CALL_010,Qwen/Qwen3-4B-Instruct-2507,success,2409,433,43.02,False



EXPERIMENT SUMMARY
Total Calls: 4
Successful: 3
Failed: 1
Average Inference Time: 49.72s

Results Directory: data/processed/summarization/model_comparison/qwen3_4b_2507

QWEN3 EXPERIMENT COMPLETED


## 19. CALL_008 Generation Retry

Retry the incomplete Qwen3 output with an increased generation limit while preserving the original prompt and model configuration.

In [26]:
import json
import time
import re
import hashlib
import pandas as pd
import torch
from pathlib import Path

print("=" * 65)
print("STAGE 04 — QWEN3 CALL_008 RETRY")
print("=" * 65)

# 1. Configuration
CALL_ID = "CALL_008"
MAX_NEW_TOKENS = 900

INPUT_DIR = Path("data/processed/summarization/input")

BASELINE_DIR = Path("data/processed/summarization/results")

Q3_OUTPUT_DIR = Path(
    "data/processed/summarization/model_comparison/qwen3_4b_2507"
)

# 2. Load transcript
with open(
    INPUT_DIR / f"{CALL_ID}.json",
    "r",
    encoding="utf-8"
) as f:
    call_data = json.load(f)

transcript = call_data["formatted_transcript"]

# 3. Load original prompt
comparison_prompt = (
    BASELINE_DIR / "system_prompt.txt"
).read_text(encoding="utf-8")

prompt_hash = hashlib.sha256(
    comparison_prompt.encode("utf-8")
).hexdigest()

messages = [
    {
        "role": "system",
        "content": comparison_prompt
    },
    {
        "role": "user",
        "content": f"Call Transcript:\n\n{transcript}"
    }
]

# 4. Tokenization
inputs = tokenizer_q3.apply_chat_template(
    messages,
    tokenize=True,
    add_generation_prompt=True,
    return_dict=True,
    return_tensors="pt"
).to(model_q3.device)

input_tokens = inputs["input_ids"].shape[-1]

print(f"\nCall ID: {CALL_ID}")
print(f"Input Tokens: {input_tokens}")
print(f"Max New Tokens: {MAX_NEW_TOKENS}")

# 5. Generate summary
print("\nGenerating summary...")

torch.cuda.synchronize()
start_time = time.perf_counter()

with torch.inference_mode():
    outputs = model_q3.generate(
        **inputs,
        max_new_tokens=MAX_NEW_TOKENS,
        do_sample=False,
        pad_token_id=tokenizer_q3.eos_token_id
    )

torch.cuda.synchronize()

inference_time = time.perf_counter() - start_time

generated_tokens = outputs[0][input_tokens:]

generated_text = tokenizer_q3.decode(
    generated_tokens,
    skip_special_tokens=True
).strip()

# 6. Remove Markdown fences if present
match = re.fullmatch(
    r"```(?:json)?[ \t]*\r?\n(.*?)\r?\n```",
    generated_text,
    flags=re.DOTALL | re.IGNORECASE
)

format_recovered = match is not None

cleaned_text = (
    match.group(1).strip()
    if format_recovered
    else generated_text
)

# 7. Validate JSON
required_fields = {
    "customer_issue",
    "agent_action",
    "final_outcome",
    "summary",
    "notes_limitations"
}

try:
    summary = json.loads(cleaned_text)

    if not isinstance(summary, dict):
        raise ValueError("Output must be a JSON object.")

    if set(summary.keys()) != required_fields:
        raise ValueError("Missing or unexpected JSON fields.")

    if not all(isinstance(v, str) for v in summary.values()):
        raise ValueError("Summary fields must be strings.")

    status = "success"

    # Save result
    result = {
        "call_id": CALL_ID,
        "model": MODEL_Q3_ID,
        "quantization": "4-bit NF4",
        "prompt_sha256": prompt_hash,
        "input_tokens": input_tokens,
        "generated_tokens": len(generated_tokens),
        "max_new_tokens": MAX_NEW_TOKENS,
        "inference_seconds": round(inference_time, 2),
        "format_recovered": format_recovered,
        "retry_reason": "Previous generation reached token limit",
        "generated_summary": summary
    }

    output_path = Q3_OUTPUT_DIR / f"{CALL_ID}_summary.json"

    with open(output_path, "w", encoding="utf-8") as f:
        json.dump(result, f, indent=2, ensure_ascii=False)

    print("\nGENERATED SUMMARY")
    print("-" * 65)

    print(json.dumps(
        summary,
        indent=2,
        ensure_ascii=False
    ))

    print(f"\nSaved: {output_path}")

except (json.JSONDecodeError, ValueError) as e:
    status = "failed"

    raw_path = Q3_OUTPUT_DIR / f"{CALL_ID}_retry_raw.txt"
    raw_path.write_text(generated_text, encoding="utf-8")

    print(f"\nJSON Validation Failed: {e}")
    print(f"Raw output saved: {raw_path}")

# 8. Update experiment metrics
metrics_path = Q3_OUTPUT_DIR / "experiment_metrics.csv"

metrics_df = pd.read_csv(metrics_path)

mask = metrics_df["call_id"] == CALL_ID

metrics_df.loc[mask, "status"] = status
metrics_df.loc[mask, "input_tokens"] = input_tokens
metrics_df.loc[mask, "generated_tokens"] = len(generated_tokens)
metrics_df.loc[mask, "inference_seconds"] = round(inference_time, 2)
metrics_df.loc[mask, "format_recovered"] = format_recovered
metrics_df.loc[mask, "max_new_tokens"] = MAX_NEW_TOKENS

metrics_df.to_csv(metrics_path, index=False)

print("\n" + "=" * 65)
print("RETRY RESULTS")
print("=" * 65)

print(f"Call ID: {CALL_ID}")
print(f"Status: {status}")
print(f"Generated Tokens: {len(generated_tokens)}")
print(f"Inference Time: {inference_time:.2f}s")

print("\nUPDATED EXPERIMENT STATUS")
print("-" * 65)

display(metrics_df)

print("\nRETRY COMPLETED")

STAGE 04 — QWEN3 CALL_008 RETRY

Call ID: CALL_008
Input Tokens: 3437
Max New Tokens: 900

Generating summary...

GENERATED SUMMARY
-----------------------------------------------------------------
{
  "customer_issue": "The customer, Bivu, recently bought a house in Noida and is seeking information about home insurance. He wants to know what a typical home insurance policy covers, especially regarding natural disasters and man-made incidents like theft or burglary. He also inquires about content (personal belongings) coverage and whether he needs to pay extra for it. Additionally, he asks about available policy options, required documents, and the claims process for theft.",
  "agent_action": "The agent, Rupa Sankar, explained that the home insurance covers structural damage due to natural disasters like earthquakes, floods, and storms. It also includes loan protection, where EMIs can be managed temporarily if the policyholder is unable to pay. The agent confirmed that personal belong

,call_id,model,status,input_tokens,generated_tokens,inference_seconds,format_recovered,max_new_tokens
0,CALL_001,Qwen/Qwen3-4B-Instruct-2507,success,1820,427,39.34,False,NaN
1,CALL_004,Qwen/Qwen3-4B-Instruct-2507,success,2657,478,50.85,False,NaN
2,CALL_008,Qwen/Qwen3-4B-Instruct-2507,success,3437,522,70.89,False,900.0
3,CALL_010,Qwen/Qwen3-4B-Instruct-2507,success,2409,433,43.02,False,NaN



RETRY COMPLETED


## 20. Model Quality Comparison

Compare Qwen2.5-3B and Qwen3-4B on four selected calls using the same evaluation criteria.

Metrics:
- Factual Accuracy
- Completeness
- Outcome Accuracy

Scores range from 1 to 5.

In [27]:
import pandas as pd
from pathlib import Path

print("=" * 65)
print("STAGE 04 — MODEL QUALITY COMPARISON")
print("=" * 65)

# 1. Define paths
BASELINE_EVAL = Path(
    "data/processed/summarization/evaluation/summary_quality_audit.csv"
)

COMPARISON_DIR = Path(
    "data/processed/summarization/model_comparison"
)

COMPARISON_DIR.mkdir(parents=True, exist_ok=True)

# 2. Load baseline evaluation
baseline_df = pd.read_csv(BASELINE_EVAL)

test_calls = [
    "CALL_001",
    "CALL_004",
    "CALL_008",
    "CALL_010"
]

baseline = baseline_df[
    baseline_df["call_id"].isin(test_calls)
].copy()

# 3. Qwen3 review scores
qwen3_scores = pd.DataFrame([
    {
        "call_id": "CALL_001",
        "q3_factual": 4,
        "q3_completeness": 5,
        "q3_outcome": 4
    },
    {
        "call_id": "CALL_004",
        "q3_factual": 4,
        "q3_completeness": 5,
        "q3_outcome": 4
    },
    {
        "call_id": "CALL_008",
        "q3_factual": 4,
        "q3_completeness": 4,
        "q3_outcome": 5
    },
    {
        "call_id": "CALL_010",
        "q3_factual": 2,
        "q3_completeness": 3,
        "q3_outcome": 1
    }
])

# 4. Merge evaluations
comparison_df = baseline.merge(
    qwen3_scores,
    on="call_id",
    how="inner",
    validate="one_to_one"
)

assert len(comparison_df) == 4

# 5. Calculate average scores
comparison_df["q25_overall"] = comparison_df[
    [
        "factual_accuracy_1_5",
        "completeness_1_5",
        "outcome_accuracy_1_5"
    ]
].mean(axis=1)

comparison_df["q3_overall"] = comparison_df[
    [
        "q3_factual",
        "q3_completeness",
        "q3_outcome"
    ]
].mean(axis=1)

comparison_df["improvement"] = (
    comparison_df["q3_overall"]
    - comparison_df["q25_overall"]
)

# 6. Display comparison
print("\nPER-CALL MODEL COMPARISON")
print("-" * 65)

display(
    comparison_df[
        [
            "call_id",
            "q25_overall",
            "q3_overall",
            "improvement"
        ]
    ].round(2)
)

# 7. Aggregate results
q25_average = comparison_df["q25_overall"].mean()
q3_average = comparison_df["q3_overall"].mean()

print("\n" + "=" * 65)
print("OVERALL MODEL COMPARISON")
print("=" * 65)

print(f"Qwen2.5-3B Average: {q25_average:.2f}/5")
print(f"Qwen3-4B Average:   {q3_average:.2f}/5")

print(
    f"Improvement: "
    f"{q3_average - q25_average:+.2f} points"
)

# 8. Save comparison
output_path = (
    COMPARISON_DIR / "model_quality_comparison.csv"
)

comparison_df.to_csv(
    output_path,
    index=False
)

print(f"\nSaved: {output_path}")

print("\nMODEL COMPARISON COMPLETED")

STAGE 04 — MODEL QUALITY COMPARISON

PER-CALL MODEL COMPARISON
-----------------------------------------------------------------


,call_id,q25_overall,q3_overall,improvement
0,CALL_001,3.00,4.33,1.33
1,CALL_004,2.00,4.33,2.33
2,CALL_008,2.00,4.33,2.33
3,CALL_010,2.33,2.00,-0.33



OVERALL MODEL COMPARISON
Qwen2.5-3B Average: 2.33/5
Qwen3-4B Average:   3.75/5
Improvement: +1.42 points

Saved: data/processed/summarization/model_comparison/model_quality_comparison.csv

MODEL COMPARISON COMPLETED


## 21. Complete Qwen3 Summarization

Generate structured summaries for the remaining eight calls using Qwen3-4B-Instruct-2507.

Preserve existing results, validate JSON outputs, and record inference metrics.

In [28]:
import json
import time
import re
import hashlib
import pandas as pd
import torch
from pathlib import Path

print("=" * 65)
print("STAGE 04 — COMPLETE QWEN3 SUMMARIZATION")
print("=" * 65)

# 1. Configuration
COMPLETED_CALLS = {
    "CALL_001",
    "CALL_004",
    "CALL_008",
    "CALL_010"
}

INPUT_DIR = Path(
    "data/processed/summarization/input"
)

BASELINE_DIR = Path(
    "data/processed/summarization/results"
)

Q3_OUTPUT_DIR = Path(
    "data/processed/summarization/model_comparison/qwen3_4b_2507"
)

Q3_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

MAX_NEW_TOKENS = 800

required_fields = {
    "customer_issue",
    "agent_action",
    "final_outcome",
    "summary",
    "notes_limitations"
}

# 2. Load selected calls
selected_calls = pd.read_csv(
    "data/processed/selected_calls.csv"
)

remaining_calls = [
    call_id
    for call_id in selected_calls["call_id"]
    if call_id not in COMPLETED_CALLS
]

# 3. Load original prompt
comparison_prompt = (
    BASELINE_DIR / "system_prompt.txt"
).read_text(encoding="utf-8")

prompt_hash = hashlib.sha256(
    comparison_prompt.encode("utf-8")
).hexdigest()

print(f"\nModel: {MODEL_Q3_ID}")
print(f"Remaining Calls: {len(remaining_calls)}")
print(f"Max New Tokens: {MAX_NEW_TOKENS}")
print(f"Prompt SHA256: {prompt_hash[:16]}...")

# 4. Process remaining calls
batch_results = []

metrics_path = (
    Q3_OUTPUT_DIR / "remaining_calls_metrics.csv"
)

for index, call_id in enumerate(remaining_calls, start=1):

    print("\n" + "=" * 65)
    print(f"[{index}/{len(remaining_calls)}] {call_id}")
    print("=" * 65)

    input_path = INPUT_DIR / f"{call_id}.json"

    output_path = (
        Q3_OUTPUT_DIR / f"{call_id}_summary.json"
    )

    # Skip existing results
    if output_path.exists():

        print("Already processed — skipped.")

        batch_results.append({
            "call_id": call_id,
            "status": "skipped_existing",
            "input_tokens": None,
            "generated_tokens": None,
            "inference_seconds": None,
            "format_recovered": None
        })

        continue

    try:

        # Load transcript
        with open(input_path, "r", encoding="utf-8") as f:
            call_data = json.load(f)

        transcript = call_data["formatted_transcript"]

        messages = [
            {
                "role": "system",
                "content": comparison_prompt
            },
            {
                "role": "user",
                "content": f"Call Transcript:\n\n{transcript}"
            }
        ]

        # 5. Tokenization
        inputs = tokenizer_q3.apply_chat_template(
            messages,
            tokenize=True,
            add_generation_prompt=True,
            return_dict=True,
            return_tensors="pt"
        ).to(model_q3.device)

        input_tokens = inputs["input_ids"].shape[-1]

        print(f"\nInput Tokens: {input_tokens}")

        if input_tokens + MAX_NEW_TOKENS > 16000:
            raise ValueError(
                f"{call_id}: Context limit exceeded."
            )

        # 6. Generation
        print("Generating summary...")

        torch.cuda.synchronize()

        start_time = time.perf_counter()

        with torch.inference_mode():

            outputs = model_q3.generate(
                **inputs,
                max_new_tokens=MAX_NEW_TOKENS,
                do_sample=False,
                pad_token_id=tokenizer_q3.eos_token_id
            )

        torch.cuda.synchronize()

        inference_time = time.perf_counter() - start_time

        generated_tokens = outputs[0][input_tokens:]

        generated_text = tokenizer_q3.decode(
            generated_tokens,
            skip_special_tokens=True
        ).strip()

        # 7. JSON recovery
        match = re.fullmatch(
            r"```(?:json)?[ \t]*\r?\n(.*?)\r?\n```",
            generated_text,
            flags=re.DOTALL | re.IGNORECASE
        )

        format_recovered = match is not None

        cleaned_text = (
            match.group(1).strip()
            if format_recovered
            else generated_text
        )

        try:

            summary = json.loads(cleaned_text)

            if not isinstance(summary, dict):
                raise ValueError("Output is not a JSON object.")

            if set(summary.keys()) != required_fields:
                raise ValueError("Invalid JSON fields.")

            if not all(
                isinstance(value, str)
                for value in summary.values()
            ):
                raise ValueError("Invalid field types.")

        except (json.JSONDecodeError, ValueError) as e:

            raw_path = (
                Q3_OUTPUT_DIR / f"{call_id}_raw.txt"
            )

            raw_path.write_text(
                generated_text,
                encoding="utf-8"
            )

            raise ValueError(
                f"JSON validation failed: {e}"
            )

        # 8. Save result
        result = {
            "call_id": call_id,
            "model": MODEL_Q3_ID,
            "quantization": "4-bit NF4",
            "prompt_sha256": prompt_hash,
            "input_tokens": input_tokens,
            "generated_tokens": len(generated_tokens),
            "max_new_tokens": MAX_NEW_TOKENS,
            "inference_seconds": round(inference_time, 2),
            "format_recovered": format_recovered,
            "generated_summary": summary
        }

        with open(
            output_path,
            "w",
            encoding="utf-8"
        ) as f:

            json.dump(
                result,
                f,
                indent=2,
                ensure_ascii=False
            )

        status = "success"

        print(f"\nStatus: {status}")
        print(f"Generated Tokens: {len(generated_tokens)}")
        print(f"Inference Time: {inference_time:.2f}s")

        batch_results.append({
            "call_id": call_id,
            "status": status,
            "input_tokens": input_tokens,
            "generated_tokens": len(generated_tokens),
            "inference_seconds": round(inference_time, 2),
            "format_recovered": format_recovered
        })

    except Exception as e:

        print(f"\nERROR: {e}")

        batch_results.append({
            "call_id": call_id,
            "status": "failed",
            "input_tokens": None,
            "generated_tokens": None,
            "inference_seconds": None,
            "format_recovered": None
        })

    finally:

        # Save progress after each call
        pd.DataFrame(batch_results).to_csv(
            metrics_path,
            index=False
        )

# 9. Final statistics
batch_df = pd.DataFrame(batch_results)

print("\n" + "=" * 65)
print("QWEN3 REMAINING CALLS RESULTS")
print("=" * 65)

display(batch_df)

# 10. Verify all summaries
saved_summaries = sorted(
    Q3_OUTPUT_DIR.glob("CALL_*_summary.json")
)

print("\n" + "=" * 65)
print("FINAL QWEN3 SUMMARY")
print("=" * 65)

print(f"Expected Summaries: 12")
print(f"Saved Summaries: {len(saved_summaries)}")

print(
    f"New Successful: "
    f"{(batch_df['status'] == 'success').sum()}"
)

print(
    f"Failed: "
    f"{(batch_df['status'] == 'failed').sum()}"
)

print(f"\nResults Directory: {Q3_OUTPUT_DIR}")

if len(saved_summaries) == 12:

    print("\nSUCCESS: All 12 Qwen3 summaries are available.")

else:

    print("\nWARNING: Some summaries are missing.")

print("\nQWEN3 SUMMARIZATION COMPLETED")

STAGE 04 — COMPLETE QWEN3 SUMMARIZATION

Model: Qwen/Qwen3-4B-Instruct-2507
Remaining Calls: 8
Max New Tokens: 800
Prompt SHA256: 9b7478816a859d7f...

[1/8] CALL_002

Input Tokens: 2920
Generating summary...

Status: success
Generated Tokens: 348
Inference Time: 41.43s

[2/8] CALL_003

Input Tokens: 3715
Generating summary...

Status: success
Generated Tokens: 408
Inference Time: 57.85s

[3/8] CALL_005

Input Tokens: 3875
Generating summary...

Status: success
Generated Tokens: 425
Inference Time: 62.90s

[4/8] CALL_006

Input Tokens: 3085
Generating summary...

Status: success
Generated Tokens: 467
Inference Time: 56.50s

[5/8] CALL_007

Input Tokens: 3849
Generating summary...

Status: success
Generated Tokens: 474
Inference Time: 68.77s

[6/8] CALL_009

Input Tokens: 2652
Generating summary...

Status: success
Generated Tokens: 428
Inference Time: 45.79s

[7/8] CALL_011

Input Tokens: 3472
Generating summary...

Status: success
Generated Tokens: 535
Inference Time: 70.21s

[8/8] CAL

,call_id,status,input_tokens,generated_tokens,inference_seconds,format_recovered
0,CALL_002,success,2920,348,41.43,False
1,CALL_003,success,3715,408,57.85,False
2,CALL_005,success,3875,425,62.90,False
3,CALL_006,success,3085,467,56.50,False
4,CALL_007,success,3849,474,68.77,False
5,CALL_009,success,2652,428,45.79,False
6,CALL_011,success,3472,535,70.21,False
7,CALL_012,success,4107,567,86.29,False



FINAL QWEN3 SUMMARY
Expected Summaries: 12
Saved Summaries: 12
New Successful: 8
Failed: 0

Results Directory: data/processed/summarization/model_comparison/qwen3_4b_2507

SUCCESS: All 12 Qwen3 summaries are available.

QWEN3 SUMMARIZATION COMPLETED


## 22. Qwen3 Quality Review

Prepare a comparative review of Qwen2.5 and Qwen3 summaries against the source transcripts.

Evaluate factual accuracy, completeness, and outcome accuracy across all 12 calls.

In [29]:
import json
import zipfile
import pandas as pd
from pathlib import Path
from google.colab import files

print("=" * 65)
print("STAGE 04 — QWEN3 QUALITY REVIEW PREPARATION")
print("=" * 65)

# 1. Directories
ROOT_DIR = Path("data/processed/summarization")

INPUT_DIR = ROOT_DIR / "input"
BASELINE_DIR = ROOT_DIR / "results"

Q3_DIR = (
    ROOT_DIR / "model_comparison/qwen3_4b_2507"
)

EVAL_DIR = ROOT_DIR / "evaluation"
EVAL_DIR.mkdir(parents=True, exist_ok=True)

# 2. Load selected calls
selected_calls = pd.read_csv(
    "data/processed/selected_calls.csv"
)

call_ids = selected_calls["call_id"].tolist()

# 3. Prepare comparison document
review_lines = [
    "# Qwen2.5 vs Qwen3 — Summary Quality Review",
    "",
    "Comparison of generated summaries against source transcripts.",
    "",
    "Evaluation criteria: Factual Accuracy, Completeness, Outcome Accuracy.",
    ""
]

for call_id in call_ids:

    # Load original transcript
    with open(
        INPUT_DIR / f"{call_id}.json",
        "r",
        encoding="utf-8"
    ) as f:
        call_data = json.load(f)

    # Load Qwen2.5 summary
    with open(
        BASELINE_DIR / f"{call_id}_summary.json",
        "r",
        encoding="utf-8"
    ) as f:
        baseline = json.load(f)

    # Load Qwen3 summary
    with open(
        Q3_DIR / f"{call_id}_summary.json",
        "r",
        encoding="utf-8"
    ) as f:
        qwen3 = json.load(f)

    # Add comparison section
    review_lines.extend([
        "",
        "---",
        f"## {call_id}",
        "",
        "### Qwen2.5-3B Summary",
        "",
        "```json",
        json.dumps(
            baseline["generated_summary"],
            indent=2,
            ensure_ascii=False
        ),
        "```",
        "",
        "### Qwen3-4B Summary",
        "",
        "```json",
        json.dumps(
            qwen3["generated_summary"],
            indent=2,
            ensure_ascii=False
        ),
        "```",
        "",
        "### Source Transcript",
        "",
        "```text",
        call_data["formatted_transcript"],
        "```",
        ""
    ])

# 4. Save review document
review_path = (
    EVAL_DIR / "qwen3_model_comparison_review.md"
)

review_path.write_text(
    "\n".join(review_lines),
    encoding="utf-8"
)

print("\nREVIEW DOCUMENT")
print("-" * 65)

print(f"Calls Included: {len(call_ids)}")
print(f"Saved: {review_path}")

# 5. Create updated backup
ZIP_PATH = Path(
    "Maharah_Summarization_Results_Qwen3.zip"
)

with zipfile.ZipFile(
    ZIP_PATH,
    "w",
    zipfile.ZIP_DEFLATED
) as zipf:

    for file_path in ROOT_DIR.rglob("*"):

        if file_path.is_file():
            zipf.write(
                file_path,
                arcname=str(file_path)
            )

# 6. Verify archive
with zipfile.ZipFile(ZIP_PATH, "r") as zipf:

    archived_files = zipf.namelist()
    corrupted = zipf.testzip()

    if corrupted is not None:
        raise RuntimeError(
            f"Corrupted file: {corrupted}"
        )

print("\nBACKUP VERIFICATION")
print("-" * 65)

print(f"Archive: {ZIP_PATH}")
print(f"Files Archived: {len(archived_files)}")
print(f"Integrity: PASSED")

# 7. Download files
print("\nDownloading review document...")
files.download(str(review_path))

print("\nDownloading updated backup...")
files.download(str(ZIP_PATH))

print("\n" + "=" * 65)
print("QWEN3 REVIEW PREPARATION COMPLETED")
print("=" * 65)

STAGE 04 — QWEN3 QUALITY REVIEW PREPARATION

REVIEW DOCUMENT
-----------------------------------------------------------------
Calls Included: 12
Saved: data/processed/summarization/evaluation/qwen3_model_comparison_review.md

BACKUP VERIFICATION
-----------------------------------------------------------------
Archive: Maharah_Summarization_Results_Qwen3.zip
Files Archived: 48
Integrity: PASSED



<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>


QWEN3 REVIEW PREPARATION COMPLETED


## 23. Final Model Evaluation

Compare Qwen2.5-3B and Qwen3-4B across all 12 calls.

Evaluate factual accuracy, completeness, and outcome accuracy using transcript-based review scores.

In [30]:
import pandas as pd
import io
from pathlib import Path
from google.colab import files

print("=" * 65)
print("STAGE 04 — FINAL MODEL EVALUATION")
print("=" * 65)

# 1. Upload Qwen3 evaluation
uploaded = files.upload()

filename = next(
    name for name in uploaded
    if name.startswith("qwen3_quality_review_scored")
    and name.endswith(".csv")
)

q3_df = pd.read_csv(
    io.BytesIO(uploaded[filename])
)

# 2. Define paths
ROOT_DIR = Path("data/processed/summarization")

EVAL_DIR = ROOT_DIR / "evaluation"
COMPARISON_DIR = ROOT_DIR / "model_comparison"

EVAL_DIR.mkdir(parents=True, exist_ok=True)
COMPARISON_DIR.mkdir(parents=True, exist_ok=True)

# 3. Load baseline evaluation
baseline_df = pd.read_csv(
    EVAL_DIR / "summary_quality_audit.csv"
)

score_columns = [
    "factual_accuracy_1_5",
    "completeness_1_5",
    "outcome_accuracy_1_5"
]

# 4. Validate evaluations
assert len(q3_df) == 12
assert q3_df["call_id"].is_unique
assert baseline_df["call_id"].is_unique

assert set(q3_df["call_id"]) == set(
    baseline_df["call_id"]
)

for column in score_columns:

    q3_df[column] = pd.to_numeric(
        q3_df[column],
        errors="raise"
    )

    if not q3_df[column].between(1, 5).all():
        raise ValueError(
            f"Invalid evaluation scores: {column}"
        )

# 5. Save Qwen3 evaluation separately
q3_eval_path = (
    EVAL_DIR / "qwen3_quality_review_scored.csv"
)

q3_df.to_csv(
    q3_eval_path,
    index=False
)

# 6. Prepare baseline scores
baseline_scores = baseline_df[
    ["call_id"] + score_columns
].copy()

baseline_scores.columns = [
    "call_id",
    "q25_factual",
    "q25_completeness",
    "q25_outcome"
]

# 7. Prepare Qwen3 scores
q3_scores = q3_df[
    ["call_id"] + score_columns
].copy()

q3_scores.columns = [
    "call_id",
    "q3_factual",
    "q3_completeness",
    "q3_outcome"
]

# 8. Merge model evaluations
comparison_df = baseline_scores.merge(
    q3_scores,
    on="call_id",
    how="inner",
    validate="one_to_one"
)

comparison_df = comparison_df.sort_values(
    "call_id"
).reset_index(drop=True)

# 9. Calculate per-call scores
comparison_df["q25_overall"] = comparison_df[
    [
        "q25_factual",
        "q25_completeness",
        "q25_outcome"
    ]
].mean(axis=1)

comparison_df["q3_overall"] = comparison_df[
    [
        "q3_factual",
        "q3_completeness",
        "q3_outcome"
    ]
].mean(axis=1)

comparison_df["improvement"] = (
    comparison_df["q3_overall"]
    - comparison_df["q25_overall"]
)

# 10. Display per-call comparison
print("\nPER-CALL MODEL COMPARISON")
print("-" * 65)

display(
    comparison_df[
        [
            "call_id",
            "q25_overall",
            "q3_overall",
            "improvement"
        ]
    ].round(2)
)

# 11. Aggregate quality metrics
metrics = [
    ("Factual Accuracy", "q25_factual", "q3_factual"),
    ("Completeness", "q25_completeness", "q3_completeness"),
    ("Outcome Accuracy", "q25_outcome", "q3_outcome"),
    ("Overall Quality", "q25_overall", "q3_overall")
]

results = []

for metric, old_col, new_col in metrics:

    old_score = comparison_df[old_col].mean()
    new_score = comparison_df[new_col].mean()

    results.append({
        "metric": metric,
        "Qwen2.5-3B": round(old_score, 2),
        "Qwen3-4B": round(new_score, 2),
        "improvement": round(
            new_score - old_score, 2
        )
    })

results_df = pd.DataFrame(results)

print("\n" + "=" * 65)
print("FINAL QUALITY COMPARISON")
print("=" * 65)

display(results_df)

# 12. Count model wins
wins = (comparison_df["improvement"] > 0).sum()
ties = (comparison_df["improvement"] == 0).sum()
losses = (comparison_df["improvement"] < 0).sum()

print(f"\nQwen3 Better: {wins}")
print(f"Equal Performance: {ties}")
print(f"Qwen2.5 Better: {losses}")

# 13. Save final results
comparison_path = (
    COMPARISON_DIR / "model_quality_comparison_12_calls.csv"
)

metrics_path = (
    COMPARISON_DIR / "final_quality_metrics.csv"
)

comparison_df.to_csv(
    comparison_path,
    index=False
)

results_df.to_csv(
    metrics_path,
    index=False
)

print("\n" + "=" * 65)
print("OUTPUT FILES")
print("=" * 65)

print(f"Qwen3 Evaluation: {q3_eval_path}")
print(f"Model Comparison: {comparison_path}")
print(f"Quality Metrics: {metrics_path}")

print("\nFINAL MODEL EVALUATION COMPLETED")

STAGE 04 — FINAL MODEL EVALUATION


Saving qwen3_quality_review_scored.csv to qwen3_quality_review_scored.csv

PER-CALL MODEL COMPARISON
-----------------------------------------------------------------


,call_id,q25_overall,q3_overall,improvement
0,CALL_001,3.00,4.33,1.33
1,CALL_002,5.00,4.33,-0.67
2,CALL_003,3.67,3.33,-0.33
3,CALL_004,2.00,4.33,2.33
4,CALL_005,3.00,3.33,0.33
5,CALL_006,2.33,3.67,1.33
6,CALL_007,4.00,4.67,0.67
7,CALL_008,2.00,4.33,2.33
8,CALL_009,4.33,4.67,0.33
9,CALL_010,2.33,2.00,-0.33



FINAL QUALITY COMPARISON


,metric,Qwen2.5-3B,Qwen3-4B,improvement
0,Factual Accuracy,3.08,3.50,0.42
1,Completeness,3.67,4.42,0.75
2,Outcome Accuracy,3.17,3.58,0.42
3,Overall Quality,3.31,3.83,0.53



Qwen3 Better: 7
Equal Performance: 1
Qwen2.5 Better: 4

OUTPUT FILES
Qwen3 Evaluation: data/processed/summarization/evaluation/qwen3_quality_review_scored.csv
Model Comparison: data/processed/summarization/model_comparison/model_quality_comparison_12_calls.csv
Quality Metrics: data/processed/summarization/model_comparison/final_quality_metrics.csv

FINAL MODEL EVALUATION COMPLETED


## 24. Final Results Export

Export the final summarization results, model comparison metrics, and quality evaluations.

**Selected Model:** Qwen3-4B-Instruct-2507

In [31]:
import json
import zipfile
from pathlib import Path
from google.colab import files

print("=" * 65)
print("STAGE 04 — FINAL RESULTS EXPORT")
print("=" * 65)

# 1. Define directories
ROOT_DIR = Path("data/processed/summarization")

Q3_DIR = (
    ROOT_DIR / "model_comparison/qwen3_4b_2507"
)

EVAL_DIR = ROOT_DIR / "evaluation"

COMPARISON_DIR = ROOT_DIR / "model_comparison"

ZIP_PATH = Path(
    "Maharah_Summarization_Results_FINAL.zip"
)

# 2. Verify required outputs
required_files = [
    EVAL_DIR / "summary_quality_audit.csv",
    EVAL_DIR / "qwen3_quality_review_scored.csv",
    COMPARISON_DIR / "model_quality_comparison_12_calls.csv",
    COMPARISON_DIR / "final_quality_metrics.csv"
]

for i in range(1, 13):
    required_files.append(
        Q3_DIR / f"CALL_{i:03d}_summary.json"
    )

missing_files = [
    path for path in required_files
    if not path.exists()
]

if missing_files:
    raise FileNotFoundError(
        f"Missing files: {missing_files}"
    )

# 3. Validate Qwen3 summaries
for i in range(1, 13):

    call_id = f"CALL_{i:03d}"

    summary_path = Q3_DIR / f"{call_id}_summary.json"

    with open(
        summary_path,
        "r",
        encoding="utf-8"
    ) as f:
        result = json.load(f)

    if result["call_id"] != call_id:
        raise ValueError(
            f"Call ID mismatch: {call_id}"
        )

    if not isinstance(
        result.get("generated_summary"),
        dict
    ):
        raise ValueError(
            f"Invalid summary: {call_id}"
        )

print("\nAll 12 Qwen3 summaries verified.")

# 4. Create final ZIP
with zipfile.ZipFile(
    ZIP_PATH,
    "w",
    zipfile.ZIP_DEFLATED
) as zipf:

    for file_path in sorted(ROOT_DIR.rglob("*")):

        if file_path.is_file():

            zipf.write(
                file_path,
                arcname=str(file_path)
            )

# 5. Verify ZIP integrity
with zipfile.ZipFile(ZIP_PATH, "r") as zipf:

    archived_files = zipf.namelist()

    corrupted_file = zipf.testzip()

    if corrupted_file is not None:
        raise RuntimeError(
            f"Corrupted file: {corrupted_file}"
        )

# 6. Display results
print("\n" + "=" * 65)
print("FINAL BACKUP VERIFICATION")
print("=" * 65)

print(f"Archive: {ZIP_PATH}")
print(f"Total Files: {len(archived_files)}")

print(
    f"Archive Size: "
    f"{ZIP_PATH.stat().st_size / 1024**2:.2f} MB"
)

print("Integrity Check: PASSED")
print("Qwen3 Summaries: 12/12")
print("Quality Evaluations: SAVED")
print("Model Comparison: SAVED")

# 7. Download ZIP
print("\nDownloading final backup...")

files.download(str(ZIP_PATH))

print("\n" + "=" * 65)
print("STAGE 04 — COMPLETED SUCCESSFULLY")
print("=" * 65)

STAGE 04 — FINAL RESULTS EXPORT

All 12 Qwen3 summaries verified.

FINAL BACKUP VERIFICATION
Archive: Maharah_Summarization_Results_FINAL.zip
Total Files: 51
Archive Size: 0.23 MB
Integrity Check: PASSED
Qwen3 Summaries: 12/12
Quality Evaluations: SAVED
Model Comparison: SAVED



<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>


STAGE 04 — COMPLETED SUCCESSFULLY
